In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#!/usr/bin/env python3
# ============================================================
# Ready-to-run: Protocol-Valid Fair Auditing of High-Confidence
# Failures under Distribution Shift (TABULAR / ACSIncome)
#
# Unified pipeline aligned to the TEXT + unified Diabetes script:
#   - Base classifier + temperature scaling
#   - Monitoring target z = 1[err & conf >= TAU_TARGET]
#   - Monitoring slice conf >= HIGH_CONF_REGION
#   - Risk scorers: entropy, energy, RED-like LR, TCP, TCP-M,
#     KOF(+stability), tcp_gate_tau, tcp_m_gate_tau,
#     kof_stab_gate_tau, random_gate_tau
#   - ACS ID/OOD mixture via train-states vs shift-states
#   - Synthetic drift stress tests scaled by rho
#   - Fair allocation policies: global / parity / bounded-topk(delta)
#   - Robust reporting: per-group counts, worst-group metrics,
#     bootstrap CI, Pareto, selection-change confirmation
#
# Also keeps the same drift-ablation strategy:
#   - full
#   - abl_no_swap
#   - abl_no_perm
#
# Sensitive attribute (eval-only): RAC1P
#
# Outputs (OUTDIR):
#   tables/: raw_* + summary_* (mean/std + bootstrap CI)
#   figs/: core curves + mitigation + Pareto
#   report.md + OUTDIR.zip
# ============================================================

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch

def set_determinism(enable: bool):
    if not enable:
        return
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

import sys
import json
import math
import random
import zipfile
import argparse
import gc
import subprocess
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# -----------------------------
# Install deps (Colab/Kaggle safe)
# -----------------------------
def pip_install(pkgs: List[str]):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install"] + pkgs)


pip_install([
    "folktables>=0.0.12",
    "numpy",
    "pandas",
    "tqdm",
    "scikit-learn>=1.2",
    "matplotlib",
    "tabulate",
])

from tqdm import tqdm
import matplotlib.pyplot as plt
import importlib.metadata as md

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

from folktables import ACSDataSource, ACSIncome


SENSITIVE_COL = "RAC1P"
ALL_FEATURES = list(ACSIncome.features)
MODEL_FEATURES = [f for f in ALL_FEATURES if str(f) != SENSITIVE_COL]
TARGET_COL = str(ACSIncome.target)


# -----------------------------
# Args
# -----------------------------
@dataclass
class Args:
    DATA_DIR: str = "./data"
    OUTDIR: str = "./out_acsincome_common_monitor"
    USE_KAGGLE_DIR: bool = True

    DEVICE: str = "auto"
    DETERMINISTIC: bool = True
    NUM_WORKERS: int = 0

    YEARS: List[int] = None
    TRAIN_STATES: List[str] = None
    SHIFT_STATES: List[str] = None
    SEEDS: List[int] = None

    MAX_PER_STATE_YEAR: int = 3500
    N_TARGET: int = 20000

    BATCH_SIZE: int = 512
    HIDDEN: int = 256
    DROPOUT: float = 0.10

    EPOCHS_CLS: int = 10
    LR_CLS: float = 2e-3
    WD: float = 1e-4
    EARLY_STOP_METRIC: str = "nll"
    EARLY_STOP_PATIENCE: int = 2
    EARLY_STOP_MIN_DELTA: float = 1e-4

    USE_TEMP_SCALING: bool = True
    TS_MAX_ITERS: int = 250

    HIGH_CONF_REGION: float = 0.80
    TAU_TARGET: float = 0.90
    ALERT_BUDGETS: List[float] = None

    RHO_VALUES: List[float] = None

    SWAP_COL_PREFS: List[str] = None
    PERM_COL_PREFS: List[str] = None
    DRIFT_BASE: float = 0.2
    DRIFT_GAIN: float = 0.80
    SWAP_TOPK: int = 60
    PERM_TOPK: int = 80
    DRIFT_ON_NONSHIFT_AT_HIGH_RHO: bool = True
    NONSHIFT_DRIFT_SCALE: float = 0.25

    TAU_LIST: List[float] = None
    KOF_HIDDEN: int = 256
    EPOCHS_DETECT: int = 4
    LR_DETECT: float = 2e-3

    EPOCHS_TCPM: int = 8
    TCPM_EARLY_STOP_PATIENCE: int = 2
    TCPM_EARLY_STOP_MIN_DELTA: float = 1e-4
    TCPM_TRAIN_ON_TAU_SLICE: bool = True
    TCPM_USE_POS_WEIGHT: bool = True
    TCPM_APPLY_TAU_GATE_AT_INFERENCE: bool = True

    USE_STABILITY: bool = True
    STAB_K: int = 4
    STAB_PERT_PROB: float = 0.35
    STAB_CONT_NOISE_STD: float = 0.12
    STAB_MAX_SAMPLES_FOR_FEAT: int = 40000

    ENABLE_FAIRNESS_MITIGATION: bool = True
    FAIRNESS_POLICIES: List[str] = None
    FAIR_TOPK_DELTAS: List[float] = None
    MITIGATE_MODELS: List[str] = None

    GROUP_MIN_N_LIST: List[int] = None
    BOOTSTRAP_N: int = 1200
    BOOTSTRAP_ALPHA: float = 0.05


@dataclass
class ExperimentConfig:
    name: str
    use_swap_drift: bool = True
    use_perm_drift: bool = True


def make_args() -> Args:
    a = Args()
    a.YEARS = [2014, 2015, 2016, 2017, 2018]
    a.TRAIN_STATES = ["CA", "TX", "FL", "NY", "PA", "IL"]
    a.SHIFT_STATES = ["OH", "GA", "NC", "MI"]
    #a.SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
    a.SEEDS = [42, 43, 44, 45, 46]
    a.RHO_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    a.TAU_LIST = [0.70, 0.80, 0.90, 0.95]
    a.ALERT_BUDGETS = [0.05, 0.10]

    a.SWAP_COL_PREFS = ["OCCP", "INDP", "SCHL", "RELP", "MAR", "COW", "POBP", "MIG"]
    a.PERM_COL_PREFS = ["INDP", "OCCP", "RELP", "SCHL", "MAR", "COW", "MIG", "POBP"]

    # CHANGED: fairness mitigation narrowed to the recommended bounded policy
    # for tcp_m_gate_tau only, with a richer delta grid.
    a.FAIRNESS_POLICIES = ["bounded"]
    a.FAIR_TOPK_DELTAS = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10]
    a.MITIGATE_MODELS = ["tcp_m_gate_tau"]

    a.GROUP_MIN_N_LIST = [1, 200, 500]

    if a.USE_KAGGLE_DIR and os.path.exists("/kaggle/working"):
        a.OUTDIR = "/kaggle/working/out_acsincome_common_monitor"
        a.DATA_DIR = "/kaggle/working/data"
    return a


args = make_args()


def experiments() -> List[ExperimentConfig]:
    return [
        ExperimentConfig(name="full", use_swap_drift=True, use_perm_drift=True),
        ExperimentConfig(name="abl_no_swap", use_swap_drift=False, use_perm_drift=True),
        ExperimentConfig(name="abl_no_perm", use_swap_drift=True, use_perm_drift=False),
    ]


# -----------------------------
# Utilities
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def set_determinism(enable: bool):
    if not enable:
        return
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def get_device(name: str):
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


def _normalize_binary_scores(y_true, y_score):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    ys = np.asarray(y_score)

    if ys.ndim == 0:
        ys = np.full(len(y_true), float(ys), dtype=float)
    elif ys.ndim == 1:
        ys = ys.astype(float).reshape(-1)
    elif ys.ndim == 2:
        if ys.shape[0] == len(y_true) and ys.shape[1] >= 2:
            ys = ys[:, 1].astype(float).reshape(-1)
        elif ys.shape[1] == len(y_true) and ys.shape[0] >= 2:
            ys = ys[1, :].astype(float).reshape(-1)
        else:
            ys = ys.reshape(-1).astype(float)
    else:
        ys = ys.reshape(-1).astype(float)

    if len(y_true) != len(ys):
        raise ValueError(
            f"Label/score length mismatch after normalization: "
            f"y_true.shape={np.asarray(y_true).shape}, y_score.shape={np.asarray(y_score).shape}, "
            f"normalized={len(y_true)} vs {len(ys)}"
        )
    return y_true, ys


def safe_auc(y_true, y_score) -> float:
    y_true, y_score = _normalize_binary_scores(y_true, y_score)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))


class ConstantProbModel:
    def __init__(self, p_pos: float):
        self.p_pos = float(np.clip(p_pos, 0.0, 1.0))

    def predict_proba(self, X):
        X = np.asarray(X)
        n = int(X.shape[0]) if X.ndim > 0 else 1
        p1 = np.full(n, self.p_pos, dtype=np.float32)
        p0 = 1.0 - p1
        return np.column_stack([p0, p1])


def dump_versions() -> Dict[str, str]:
    pkgs = ["folktables", "scikit-learn", "torch", "pandas", "numpy", "matplotlib"]
    out = {}
    for p in pkgs:
        try:
            out[p] = md.version(p)
        except Exception:
            out[p] = "unknown"
    return out


def p_drift_from_rho(rho: float) -> float:
    return float(np.clip(args.DRIFT_BASE + args.DRIFT_GAIN * float(rho), 0.0, 1.0))


def ensure_outdirs(outdir: str):
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "tables"), exist_ok=True)
    os.makedirs(os.path.join(outdir, "figs"), exist_ok=True)


def mean_std_ci_over_seeds(
    df: pd.DataFrame,
    group_cols: List[str],
    metric_cols: List[str],
    seed_col: str = "seed",
    n_boot: int = 1000,
    alpha: float = 0.05,
) -> pd.DataFrame:
    out_rows = []
    base = df.copy()
    base_seed = base.groupby(group_cols + [seed_col], as_index=False)[metric_cols].mean()

    nb = int(n_boot) if (n_boot is not None and int(n_boot) > 0) else 0
    rng = np.random.default_rng(12345)

    for keys, sub in base_seed.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {c: v for c, v in zip(group_cols, keys)}

        for m in metric_cols:
            vals = sub[m].to_numpy(dtype=float)
            row[f"{m}_mean"] = float(np.nanmean(vals)) if np.isfinite(vals).any() else float("nan")
            row[f"{m}_std"] = float(np.nanstd(vals, ddof=1)) if np.isfinite(vals).sum() > 1 else float("nan")

            seed_vals = sub.groupby(seed_col)[m].mean().to_numpy(dtype=float)
            finite = seed_vals[np.isfinite(seed_vals)]
            s_eff = int(len(finite))
            row[f"{m}_n_seed_finite"] = s_eff

            if nb < 1 or s_eff < 2:
                row[f"{m}_ci_low"] = float("nan")
                row[f"{m}_ci_high"] = float("nan")
            else:
                boot = np.empty(nb, dtype=float)
                for bi in range(nb):
                    idx = rng.integers(0, s_eff, size=s_eff)
                    boot[bi] = float(np.mean(finite[idx]))
                row[f"{m}_ci_low"] = float(np.quantile(boot, alpha / 2))
                row[f"{m}_ci_high"] = float(np.quantile(boot, 1 - alpha / 2))
        out_rows.append(row)

    return pd.DataFrame(out_rows)


# -----------------------------
# Dataset / shift statistics reporting (reviewer-alignment; consistent with diabetes script)
# NOTE: ACSIncome's sensitive attribute (RAC1P) is multi-valued, unlike the
# binary male/female group in the diabetes script. This generalizes
# compute_split_stats() to loop over all observed group ids while preserving
# the same field-naming convention (n_group{g}, group{g}_frac, n_pos_group{g}, ...).
# -----------------------------
def compute_split_stats(name: str, rho, y: np.ndarray, g: np.ndarray,
                         z: Optional[np.ndarray] = None, high_mask: Optional[np.ndarray] = None) -> Dict[str, object]:
    """
    Records dataset sizes, group distributions, class imbalance,
    and (when available) high-confidence-failure / positive-target counts
    for a given split or shift condition (rho). Generalized to an arbitrary
    number of groups (RAC1P categories) rather than assuming binary groups.
    """
    y = np.asarray(y).reshape(-1).astype(np.int64)
    g = np.asarray(g).reshape(-1).astype(np.int64)
    n_total = int(len(y))

    n_pos = int(y.sum())
    n_neg = n_total - n_pos

    stats: Dict[str, object] = {
        "split": name,
        "rho": (float(rho) if rho is not None else float("nan")),
        "n_total": n_total,
        "n_pos_target": n_pos,
        "n_neg_target": int(n_neg),
        "pos_rate": float(n_pos / n_total) if n_total > 0 else float("nan"),
        "class_imbalance_ratio": float(n_neg / max(n_pos, 1)),
    }

    uniq_g = np.unique(g)
    for gg in uniq_g:
        gg = int(gg)
        m = (g == gg)
        n_g = int(m.sum())
        stats[f"n_group{gg}"] = n_g
        stats[f"group{gg}_frac"] = float(n_g / n_total) if n_total > 0 else float("nan")
        stats[f"n_pos_group{gg}"] = int(y[m].sum()) if n_g > 0 else 0

    if high_mask is not None:
        high_mask = np.asarray(high_mask).reshape(-1).astype(bool)
        n_high = int(high_mask.sum())
        stats["n_high_conf"] = n_high
        stats["high_conf_frac"] = float(n_high / n_total) if n_total > 0 else float("nan")
        for gg in uniq_g:
            gg = int(gg)
            stats[f"n_high_conf_group{gg}"] = int((high_mask & (g == gg)).sum())
    else:
        n_high = None

    if z is not None:
        z = np.asarray(z).reshape(-1).astype(np.int64)
        stats["n_high_conf_failures"] = int(z.sum())
        stats["failure_rate_in_high_conf"] = float(z.mean()) if len(z) > 0 else float("nan")
        if high_mask is not None and n_high is not None and n_high > 0:
            z_high = z[high_mask]
            g_high = g[high_mask]
            for gg in uniq_g:
                gg = int(gg)
                n_hg = int((g_high == gg).sum())
                n_fail_g = int(z_high[g_high == gg].sum()) if n_hg > 0 else 0
                stats[f"n_failures_group{gg}"] = n_fail_g
                stats[f"failure_rate_group{gg}"] = float(n_fail_g / n_hg) if n_hg > 0 else float("nan")

    return stats


def dump_dataset_statistics(all_stats: List[Dict[str, object]], outdir: str) -> pd.DataFrame:
    df = pd.DataFrame(all_stats)
    df.to_csv(os.path.join(outdir, "tables", "dataset_shift_statistics.csv"), index=False)
    return df

# -----------------------------
# Data loading
# -----------------------------
def load_acs_income_year(year: int, states: List[str], root_dir: str, max_per_state: int, seed: int) -> Dict[str, object]:
    try:
        ds = ACSDataSource(survey_year=str(year), horizon="1-Year", survey="person", root_dir=root_dir)
    except TypeError:
        ds = ACSDataSource(survey_year=str(year), horizon="1-Year", survey="person")

    rng = np.random.default_rng(seed + year)
    X_list, y_list, st_list, g_list = [], [], [], []

    for abbr in states:
        acs_state = ds.get_data(states=[abbr], download=True)
        Xs = acs_state[MODEL_FEATURES].copy()

        if SENSITIVE_COL not in acs_state.columns:
            raise RuntimeError(f"Sensitive column {SENSITIVE_COL} not found.")
        gs = acs_state[SENSITIVE_COL].to_numpy()

        if TARGET_COL not in acs_state.columns:
            raise RuntimeError(f"Target column {TARGET_COL} not found.")
        y_raw = acs_state[TARGET_COL].to_numpy()
        ys = (y_raw > 50000).astype(np.int64).reshape(-1)

        good = np.ones(len(ys), dtype=bool)
        for c in MODEL_FEATURES:
            col = Xs[c]
            if col.isna().any():
                good &= (~col.isna()).to_numpy()
        if np.isnan(y_raw).any():
            good &= (~np.isnan(y_raw))
        if np.isnan(gs).any():
            good &= (~np.isnan(gs))

        Xs = Xs.loc[good].reset_index(drop=True)
        ys = ys[good].reshape(-1)
        gs = gs[good].reshape(-1)

        if max_per_state and len(ys) > max_per_state:
            idx = rng.choice(len(ys), size=max_per_state, replace=False)
            idx = np.sort(idx)
            Xs = Xs.iloc[idx].reset_index(drop=True)
            ys = ys[idx].reshape(-1)
            gs = gs[idx].reshape(-1)

        X_list.append(Xs)
        y_list.append(ys)
        g_list.append(gs.astype(np.int64))
        st_list.append(np.array([abbr] * len(ys), dtype=object))

    return {
        "X": pd.concat(X_list, axis=0).reset_index(drop=True),
        "y": np.concatenate(y_list).astype(np.int64).reshape(-1),
        "g": np.concatenate(g_list).astype(np.int64).reshape(-1),
        "st": np.concatenate(st_list).astype(object).reshape(-1),
    }


def filter_by_states(d: Dict[str, object], states_keep: List[str]) -> Dict[str, object]:
    st = np.asarray(d["st"]).reshape(-1).astype(object)
    keep = np.isin(st, np.asarray(states_keep).astype(object))
    return {
        "X": d["X"].loc[keep].reset_index(drop=True),
        "y": np.asarray(d["y"])[keep].reshape(-1).astype(np.int64),
        "g": np.asarray(d["g"])[keep].reshape(-1).astype(np.int64),
        "st": st[keep].reshape(-1),
    }


def slice_by_idx(d: Dict[str, object], idx: np.ndarray) -> Dict[str, object]:
    return {
        "X": d["X"].iloc[idx].reset_index(drop=True),
        "y": np.asarray(d["y"])[idx].reshape(-1).astype(np.int64),
        "g": np.asarray(d["g"])[idx].reshape(-1).astype(np.int64),
        "st": np.asarray(d["st"])[idx].reshape(-1).astype(object),
    }


# -----------------------------
# Drift stress tests
# -----------------------------
def pick_existing_col(preferences: List[str], X_df: pd.DataFrame, exclude: set = None) -> str:
    exclude = exclude or set()
    for c in preferences:
        if c in X_df.columns and c not in exclude:
            return c
    best, best_u = None, -1
    for c in X_df.columns:
        if c in exclude:
            continue
        u = int(X_df[c].astype(str).nunique(dropna=True))
        if 10 <= u <= 2000 and u > best_u:
            best_u = u
            best = c
    return best if best is not None else X_df.columns[0]


def build_swap_mapping_label_informed(X_df: pd.DataFrame, y: np.ndarray, col: str, top_k: int) -> Dict[str, str]:
    df = pd.DataFrame({"v": X_df[col].astype(str), "y": np.asarray(y).reshape(-1).astype(int)})
    grp = df.groupby("v")["y"].mean().sort_values()
    if len(grp) < 2:
        return {}
    k = min(int(top_k), len(grp) // 2)
    low = grp.index[:k].tolist()
    high = grp.index[-k:].tolist()
    m = {}
    for a, b in zip(low, high):
        m[a] = b
        m[b] = a
    return m


def build_perm_mapping_label_free(series: pd.Series, top_k: int, seed: int) -> Dict[str, str]:
    s = series.astype(str)
    freq = s.value_counts()
    keys = freq.index[:min(int(top_k), len(freq))].tolist()
    rng = np.random.default_rng(seed)
    perm = keys.copy()
    rng.shuffle(perm)
    return dict(zip(keys, perm))


def apply_mapping_drift(X_df: pd.DataFrame, col: str, mapping: Dict[str, str], prob: float, seed: int) -> Tuple[pd.DataFrame, np.ndarray]:
    prob = float(np.clip(prob, 0.0, 1.0))
    if prob <= 0 or not mapping or (col not in X_df.columns):
        return X_df.copy(), np.zeros(len(X_df), dtype=np.int8)
    rng = np.random.default_rng(seed)
    X = X_df.copy()
    s = X[col].astype(str)
    in_map = s.isin(mapping.keys()).to_numpy()
    flip = rng.random(len(X)) < prob
    change = in_map & flip
    if change.any():
        s2 = s.copy()
        s2.loc[change] = s2.loc[change].map(mapping).fillna(s2.loc[change])
        X[col] = s2.values
    return X, change.astype(np.int8)


def apply_combined_drift(
    cfg: ExperimentConfig,
    X_df: pd.DataFrame,
    rho: float,
    seed: int,
    swap_col: str,
    perm_col: str,
    swap_map: Dict[str, str],
    perm_map: Dict[str, str],
    p_override: Optional[float] = None,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    p = p_drift_from_rho(rho) if p_override is None else float(np.clip(p_override, 0.0, 1.0))
    if cfg.use_swap_drift:
        X1, m_swap = apply_mapping_drift(X_df, swap_col, swap_map, prob=p, seed=seed + 1)
    else:
        X1, m_swap = X_df.copy(), np.zeros(len(X_df), dtype=np.int8)
    if cfg.use_perm_drift:
        X2, m_perm = apply_mapping_drift(X1, perm_col, perm_map, prob=p, seed=seed + 2)
    else:
        X2, m_perm = X1, np.zeros(len(X1), dtype=np.int8)
    return X2, {"swap": m_swap, "perm": m_perm, "any": ((m_swap + m_perm) > 0).astype(np.int8)}


# -----------------------------
# Preprocessing
# -----------------------------
def infer_cat_cont_cols(X_df: pd.DataFrame, force_cat_cols: List[str]) -> Tuple[List[str], List[str]]:
    default_cont = set(["AGEP", "WKHP"])
    cont = [c for c in X_df.columns if c in default_cont]
    if len(cont) == 0:
        cont = []
    for c in X_df.columns:
        s = X_df[c]
        if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > 100:
            cont.append(c)
    cat = [c for c in X_df.columns if c not in cont]
    for dc in force_cat_cols:
        if dc in cont:
            cont.remove(dc)
            cat.append(dc)
    return cat, cont


def fit_encoders(X_train: pd.DataFrame, cat_cols: List[str], cont_cols: List[str]):
    cat_maps, cat_cards = {}, {}
    for c in cat_cols:
        vals = pd.Series(X_train[c].astype(str).unique())
        mapping = {v: (i + 1) for i, v in enumerate(vals.tolist())}
        cat_maps[c] = mapping
        cat_cards[c] = len(mapping) + 1
    if len(cont_cols):
        cont_mean = X_train[cont_cols].astype(np.float32).mean(axis=0).to_numpy().astype(np.float32)
        cont_std = X_train[cont_cols].astype(np.float32).std(axis=0).replace(0.0, 1.0).to_numpy().astype(np.float32)
    else:
        cont_mean = np.array([], dtype=np.float32)
        cont_std = np.array([], dtype=np.float32)
    return cat_maps, cat_cards, cont_mean, cont_std


def encode_split(
    X_df: pd.DataFrame,
    y: np.ndarray,
    cat_cols: List[str],
    cont_cols: List[str],
    cat_maps: Dict[str, Dict[str, int]],
    cont_mean: np.ndarray,
    cont_std: np.ndarray,
):
    y = np.asarray(y).reshape(-1).astype(np.int64)

    if len(cat_cols):
        Xc = np.zeros((len(X_df), len(cat_cols)), dtype=np.int64)
        for j, c in enumerate(cat_cols):
            mapping = cat_maps[c]
            Xc[:, j] = X_df[c].astype(str).map(lambda v: mapping.get(v, 0)).to_numpy(dtype=np.int64)
    else:
        Xc = np.zeros((len(X_df), 0), dtype=np.int64)

    if len(cont_cols):
        Xraw = X_df[cont_cols].astype(np.float32).to_numpy()
        Xn = ((Xraw - cont_mean) / cont_std).astype(np.float32)
    else:
        Xn = np.zeros((len(X_df), 0), dtype=np.float32)
    return Xc, Xn, y


class TabDataset(Dataset):
    def __init__(self, Xc, Xn, y):
        self.Xc = torch.from_numpy(np.asarray(Xc))
        self.Xn = torch.from_numpy(np.asarray(Xn))
        self.y = torch.from_numpy(np.asarray(y).reshape(-1).astype(np.int64)).long()

    def __len__(self):
        return int(self.y.numel())

    def __getitem__(self, i):
        return self.Xc[i], self.Xn[i], self.y[i]


# -----------------------------
# Base model
# -----------------------------
def emb_dim(card: int) -> int:
    return int(min(64, max(4, round(math.sqrt(card) * 2))))


class TabMLP(nn.Module):
    def __init__(self, cat_cards: Dict[str, int], n_cont: int, hidden: int, dropout: float):
        super().__init__()
        self.cat_cols = list(cat_cards.keys())
        self.embs = nn.ModuleList()
        emb_dims = []
        for c in self.cat_cols:
            card = cat_cards[c]
            d = emb_dim(card)
            emb_dims.append(d)
            self.embs.append(nn.Embedding(card, d))
        in_dim = sum(emb_dims) + n_cont
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, 2)

    def forward(self, Xc, Xn):
        parts = [emb(Xc[:, j]) for j, emb in enumerate(self.embs)]
        if len(parts):
            z = torch.cat(parts, dim=-1)
            if Xn.numel():
                z = torch.cat([z, Xn], dim=-1)
        else:
            z = Xn
        h = F.relu(self.fc1(z), inplace=True)
        h = self.drop(h)
        h = F.relu(self.fc2(h), inplace=True)
        h = self.drop(h)
        logits = self.out(h)
        return h, logits


def probs_from_logits(logits):
    return F.softmax(logits, dim=-1)


def conf_from_logits(logits):
    return probs_from_logits(logits).max(dim=-1).values


def entropy_from_logits(logits):
    p = probs_from_logits(logits).clamp_min(1e-8)
    return -(p * p.log()).sum(dim=-1)


def margin_from_logits(logits):
    top2 = torch.topk(logits, k=2, dim=-1).values
    return top2[:, 0] - top2[:, 1]


def energy_from_logits(logits):
    return -torch.logsumexp(logits, dim=-1)


class TempScaler(nn.Module):
    def __init__(self, init_T: float = 1.0):
        super().__init__()
        self.logT = nn.Parameter(torch.tensor([math.log(init_T)], dtype=torch.float32))

    def forward(self, logits):
        T = torch.exp(self.logT).clamp(1e-3, 1e3)
        return logits / T


@torch.no_grad()
def eval_cls(model, loader, device, temp_scaler: Optional[TempScaler] = None):
    model.eval()
    if temp_scaler is not None:
        temp_scaler.eval()
    tot, correct, nll_sum = 0, 0, 0.0
    for Xc, Xn, y in loader:
        Xc, Xn, y = Xc.to(device), Xn.to(device), y.to(device)
        _, logits = model(Xc, Xn)
        if temp_scaler is not None:
            logits = temp_scaler(logits)
        nll_sum += float(F.cross_entropy(logits, y, reduction="sum").item())
        correct += int((logits.argmax(dim=-1) == y).sum().item())
        tot += int(y.numel())
    return {"acc": correct / max(1, tot), "nll": nll_sum / max(1, tot)}


def train_cls_with_early_stopping(model, train_loader, val_loader, device):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.LR_CLS, weight_decay=args.WD)

    best_state = None
    best_epoch = 0
    bad = 0

    metric = args.EARLY_STOP_METRIC.lower().strip()
    if metric not in ["nll", "acc"]:
        metric = "nll"

    if metric == "nll":
        best_val = float("inf")

        def is_better(v):
            return (best_val - v) > float(args.EARLY_STOP_MIN_DELTA)
    else:
        best_val = -float("inf")

        def is_better(v):
            return (v - best_val) > float(args.EARLY_STOP_MIN_DELTA)

    for ep in range(1, int(args.EPOCHS_CLS) + 1):
        model.train()
        losses = []
        for Xc, Xn, y in tqdm(train_loader, desc=f"[CLS] ep {ep}/{args.EPOCHS_CLS}", leave=False):
            Xc, Xn, y = Xc.to(device), Xn.to(device), y.to(device)
            _, logits = model(Xc, Xn)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        m = eval_cls(model, val_loader, device)
        print(f"[CLS] ep={ep:02d} train_loss={np.mean(losses):.4f} val_acc={m['acc']:.4f} val_nll={m['nll']:.4f}")

        cur = float(m[metric])
        if is_better(cur):
            best_val = cur
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = ep
            bad = 0
        else:
            bad += 1
            if bad >= int(args.EARLY_STOP_PATIENCE):
                print(f"[CLS] Early stopping at ep={ep:02d}. Best ep={best_epoch:02d} best_{metric}={best_val:.6f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[CLS] Restored best checkpoint from ep={best_epoch:02d} (best_{metric}={best_val:.6f})")
    return model


def fit_temperature_scaler(model, calib_loader, device) -> TempScaler:
    scaler = TempScaler(init_T=1.0).to(device)
    opt = torch.optim.LBFGS(scaler.parameters(), max_iter=args.TS_MAX_ITERS, line_search_fn="strong_wolfe")

    model.eval()
    logits_list, y_list = [], []
    with torch.no_grad():
        for Xc, Xn, y in calib_loader:
            Xc, Xn, y = Xc.to(device), Xn.to(device), y.to(device)
            _, logits = model(Xc, Xn)
            logits_list.append(logits.detach())
            y_list.append(y)
    logits_all = torch.cat(logits_list, dim=0)
    y_all = torch.cat(y_list, dim=0)

    def closure():
        opt.zero_grad(set_to_none=True)
        scaled = scaler(logits_all)
        loss = F.cross_entropy(scaled, y_all)
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        T = float(torch.exp(scaler.logT).item())
        print(f"[TempScaling] fitted T={T:.4f}")
    return scaler


@torch.no_grad()
def make_feat_batch(model, Xc, Xn, temp_scaler: Optional[TempScaler] = None):
    h, logits_raw = model(Xc, Xn)
    logits = temp_scaler(logits_raw) if temp_scaler is not None else logits_raw
    conf = conf_from_logits(logits)
    ent = entropy_from_logits(logits)
    marg = margin_from_logits(logits)
    feats = torch.cat([h, logits, marg.unsqueeze(-1), ent.unsqueeze(-1)], dim=-1)
    return feats, logits_raw, logits, conf, marg, ent
# -----------------------------
# Stability features
# -----------------------------
class KnowledgeOps:
    def __init__(self, cat_cols: List[str], cont_cols: List[str], X_ref: pd.DataFrame, seed: int):
        self.cat_cols = list(cat_cols)
        self.cont_cols = list(cont_cols)
        self.rng = np.random.default_rng(seed + 2025)

        self.cat_perm_maps: Dict[str, Dict[str, str]] = {}
        for c in self.cat_cols:
            freq = X_ref[c].astype(str).value_counts()
            keys = freq.index[:min(120, len(freq))].tolist()
            perm = keys.copy()
            self.rng.shuffle(perm)
            self.cat_perm_maps[c] = dict(zip(keys, perm))

        self.cont_min, self.cont_max = {}, {}
        for c in self.cont_cols:
            s = X_ref[c].astype(float)
            self.cont_min[c] = float(np.nanpercentile(s, 0.5))
            self.cont_max[c] = float(np.nanpercentile(s, 99.5))

    def perturb(self, X_df: pd.DataFrame, prob: float, cont_std: float) -> pd.DataFrame:
        Xp = X_df.copy()
        n = len(Xp)
        prob = float(np.clip(prob, 0.0, 1.0))

        for c in self.cat_cols:
            mp = self.cat_perm_maps.get(c, {})
            if not mp:
                continue
            s = Xp[c].astype(str)
            in_map = s.isin(mp.keys()).to_numpy()
            flip = self.rng.random(n) < prob
            change = in_map & flip
            if change.any():
                s2 = s.copy()
                s2.loc[change] = s2.loc[change].map(mp).fillna(s2.loc[change])
                Xp[c] = s2.values

        for c in self.cont_cols:
            if n == 0:
                continue
            x = Xp[c].astype(float).to_numpy()
            noise = self.rng.normal(0.0, cont_std, size=n)
            x2 = np.clip(x + noise, self.cont_min[c], self.cont_max[c])
            Xp[c] = x2
        return Xp


@torch.no_grad()
def compute_stability_features(
    model: nn.Module,
    temp_scaler: Optional[TempScaler],
    X_df: pd.DataFrame,
    y: np.ndarray,
    kops: KnowledgeOps,
    cat_cols: List[str],
    cont_cols: List[str],
    cat_maps: Dict[str, Dict[str, int]],
    cont_mean: np.ndarray,
    cont_std: np.ndarray,
    device: torch.device,
    K: int,
    prob: float,
    cont_noise_std: float,
    batch_size: int,
) -> np.ndarray:
    Xc0, Xn0, y0 = encode_split(X_df, y, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    loader0 = DataLoader(TabDataset(Xc0, Xn0, y0), batch_size=batch_size, shuffle=False, num_workers=0)

    model.eval()
    if temp_scaler is not None:
        temp_scaler.eval()

    conf0_all, pred0_all = [], []
    for Xc, Xn, _ in loader0:
        Xc, Xn = Xc.to(device), Xn.to(device)
        _, _, logits, conf, _, _ = make_feat_batch(model, Xc, Xn, temp_scaler=temp_scaler)
        conf0_all.append(conf.cpu().numpy())
        pred0_all.append(logits.argmax(dim=-1).cpu().numpy())
    conf0 = np.concatenate(conf0_all).reshape(-1)
    pred0 = np.concatenate(pred0_all).reshape(-1)

    conf_ks, flip_ks, margin_ks, ent_ks = [], [], [], []
    for _ in range(K):
        Xp = kops.perturb(X_df, prob=prob, cont_std=cont_noise_std)
        Xc, Xn, yy = encode_split(Xp, y, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
        ld = DataLoader(TabDataset(Xc, Xn, yy), batch_size=batch_size, shuffle=False, num_workers=0)

        confk_all, predk_all, marg_all, ent_all = [], [], [], []
        for Xc_b, Xn_b, _ in ld:
            Xc_b, Xn_b = Xc_b.to(device), Xn_b.to(device)
            _, _, logits, confk, margk, entk = make_feat_batch(model, Xc_b, Xn_b, temp_scaler=temp_scaler)
            confk_all.append(confk.cpu().numpy())
            predk_all.append(logits.argmax(dim=-1).cpu().numpy())
            marg_all.append(margk.cpu().numpy())
            ent_all.append(entk.cpu().numpy())

        confk = np.concatenate(confk_all).reshape(-1)
        predk = np.concatenate(predk_all).reshape(-1)
        margk = np.concatenate(marg_all).reshape(-1)
        entk = np.concatenate(ent_all).reshape(-1)

        conf_ks.append(confk)
        flip_ks.append((predk != pred0).astype(np.float32))
        margin_ks.append(margk)
        ent_ks.append(entk)

    conf_ks = np.stack(conf_ks, axis=1)
    flip_ks = np.stack(flip_ks, axis=1)
    margin_ks = np.stack(margin_ks, axis=1)
    ent_ks = np.stack(ent_ks, axis=1)

    conf_drop_max = (conf0 - np.min(conf_ks, axis=1)).astype(np.float32)
    flip_rate = np.mean(flip_ks, axis=1).astype(np.float32)
    margin_std = np.std(margin_ks, axis=1).astype(np.float32)
    ent_mean = np.mean(ent_ks, axis=1).astype(np.float32)
    return np.stack([conf_drop_max, flip_rate, margin_std, ent_mean], axis=1)


# -----------------------------
# Monitoring target + baselines
# -----------------------------
def overconfident_error_label(err: np.ndarray, conf: np.ndarray, tau: float) -> np.ndarray:
    err = np.asarray(err).astype(int).reshape(-1)
    conf = np.asarray(conf).astype(float).reshape(-1)
    return ((err == 1) & (conf >= float(tau))).astype(int)


def fit_red_like_lr(features: np.ndarray, z_target: np.ndarray, seed: int):
    X = features.astype(np.float32)
    y = np.asarray(z_target).astype(int).reshape(-1)
    if len(np.unique(y)) < 2:
        p = float(np.mean(y)) if len(y) else 0.0
        print(f"[RED-like LR] single-class target encountered; using constant predictor p={p:.6f}")
        return ConstantProbModel(p)
    X_tr, X_ev, y_tr, y_ev = train_test_split(X, y, test_size=0.25, random_state=seed + 17, stratify=y)
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=1)
    lr.fit(X_tr, y_tr)
    print(f"[RED-like LR] AUC={safe_auc(y_ev, lr.predict_proba(X_ev)[:, 1]):.4f}")
    return lr


def calibrate_isotonic(scores: np.ndarray, labels: np.ndarray) -> Optional[IsotonicRegression]:
    y, s = _normalize_binary_scores(labels, scores)
    m = np.isfinite(s)
    y = y[m]
    s = s[m]
    if len(y) < 50 or len(np.unique(y)) < 2:
        return None
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(s, y)
    return iso


# -----------------------------
# Detectors: TCP / TCP-M / KOF
# -----------------------------
class TCPHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class KOFDetector(nn.Module):
    def __init__(self, in_dim: int, hidden: int, n_tau: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
        )
        self.head = nn.Linear(hidden, n_tau)

    def forward(self, x):
        return self.head(self.net(x))


def make_z_multi_tau(err: torch.Tensor, conf: torch.Tensor, tau_list: List[float]) -> torch.Tensor:
    zs = [(((err > 0.5) & (conf >= float(t))).float()) for t in tau_list]
    return torch.stack(zs, dim=1)


@torch.no_grad()
def kof_head_probs(kof_logits: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-kof_logits))


def fit_tau_combo_lr(head_probs: np.ndarray, z_target: np.ndarray, seed: int):
    X = head_probs.astype(np.float32)
    y = np.asarray(z_target).astype(int).reshape(-1)
    if len(np.unique(y)) < 2:
        p = float(np.mean(y)) if len(y) else 0.0
        print(f"[TauCombo LR] single-class target encountered; using constant predictor p={p:.6f}")
        return ConstantProbModel(p)
    X_tr, X_ev, y_tr, y_ev = train_test_split(X, y, test_size=0.25, random_state=seed + 101, stratify=y)
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=1)
    lr.fit(X_tr, y_tr)
    print(f"[TauCombo LR] AUC={safe_auc(y_ev, lr.predict_proba(X_ev)[:, 1]):.4f}")
    return lr


def train_tcp_correctness(backbone: nn.Module, tcp: nn.Module, feats_loader, device):
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    tcp.to(device)
    opt = torch.optim.AdamW(tcp.parameters(), lr=args.LR_DETECT)
    bce = nn.BCEWithLogitsLoss()

    for ep in range(1, int(args.EPOCHS_DETECT) + 1):
        tcp.train()
        losses = []
        for feats, y, pred in tqdm(feats_loader, desc=f"[TCP] ep {ep}/{args.EPOCHS_DETECT}", leave=False):
            feats = feats.to(device)
            y = y.to(device)
            pred = pred.to(device)
            corr = (pred == y).float()
            loss = bce(tcp(feats), corr)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))
        print(f"[TCP] ep={ep:02d} loss={np.mean(losses):.6f}")


def _sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


@torch.no_grad()
def _tcpm_val_auc(tcpm: nn.Module, val_feats: torch.Tensor, val_conf: np.ndarray, val_z: np.ndarray, device: torch.device) -> float:
    tcpm.eval()
    logits = tcpm(val_feats.to(device)).detach().cpu().numpy().reshape(-1)
    risk = _sigmoid_np(logits).astype(np.float32)
    if args.TCPM_APPLY_TAU_GATE_AT_INFERENCE:
        risk[val_conf < float(args.TAU_TARGET)] = -1e9
    high = val_conf >= float(args.HIGH_CONF_REGION)
    return safe_auc(val_z[high], risk[high])


def train_tcp_monitoring_strong(
    backbone: nn.Module,
    tcpm: nn.Module,
    feats_train: torch.Tensor,
    z_train: torch.Tensor,
    device: torch.device,
    val_feats: Optional[torch.Tensor] = None,
    val_conf: Optional[np.ndarray] = None,
    val_z: Optional[np.ndarray] = None,
):
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False

    tcpm.to(device)
    opt = torch.optim.AdamW(tcpm.parameters(), lr=args.LR_DETECT)

    if args.TCPM_USE_POS_WEIGHT:
        z_np = z_train.detach().cpu().numpy().astype(np.int64)
        pos = float(z_np.sum())
        neg = float(len(z_np) - pos)
        pw = (neg / max(pos, 1.0)) if pos > 0 else 1.0
        pw = float(np.clip(pw, 1.0, 1e4))
        pos_weight = torch.tensor([pw], dtype=torch.float32, device=device)
        crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        print(f"[TCP-M] pos={int(pos)} neg={int(neg)} pos_weight={pw:.3f} (train slice)")
    else:
        crit = nn.BCEWithLogitsLoss()

    class FeatsForTCPM(Dataset):
        def __init__(self, feats: torch.Tensor, z: torch.Tensor):
            self.feats = feats
            self.z = z

        def __len__(self):
            return int(self.z.numel())

        def __getitem__(self, i):
            return self.feats[i], self.z[i]

    loader = DataLoader(FeatsForTCPM(feats_train, z_train), batch_size=1024, shuffle=True)

    best_state = None
    best_auc = -1.0
    bad = 0

    for ep in range(1, int(args.EPOCHS_TCPM) + 1):
        tcpm.train()
        losses = []
        for feats, z in tqdm(loader, desc=f"[TCP-M] ep {ep}/{args.EPOCHS_TCPM}", leave=False):
            feats = feats.to(device)
            z = z.to(device).float()
            logits = tcpm(feats)
            loss = crit(logits, z)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        auc_val = float("nan")
        if val_feats is not None and val_conf is not None and val_z is not None:
            auc_val = _tcpm_val_auc(tcpm, val_feats, val_conf, val_z, device)
        print(f"[TCP-M] ep={ep:02d} loss={np.mean(losses) if losses else float('nan'):.6f} val_HC_AUC={auc_val:.4f}")

        if np.isfinite(auc_val) and (auc_val - best_auc) > float(args.TCPM_EARLY_STOP_MIN_DELTA):
            best_auc = auc_val
            best_state = {k: v.detach().cpu().clone() for k, v in tcpm.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= int(args.TCPM_EARLY_STOP_PATIENCE):
                print(f"[TCP-M] Early stopping at ep={ep:02d}, best_val_HC_AUC={best_auc:.4f}")
                break

    if best_state is not None:
        tcpm.load_state_dict(best_state)
        print(f"[TCP-M] Restored best checkpoint (val_HC_AUC={best_auc:.4f})")


@torch.no_grad()
def compute_kof_pos_weight(pool_y: torch.Tensor, pool_logits_cal: torch.Tensor) -> np.ndarray:
    conf = conf_from_logits(pool_logits_cal)
    pred = pool_logits_cal.argmax(dim=-1)
    err = (pred != pool_y).float()
    z = make_z_multi_tau(err, conf, args.TAU_LIST).cpu().numpy()
    z_sum = z.sum(axis=0)
    n_sum = z.shape[0]
    pos = np.maximum(z_sum, 1.0)
    neg = np.maximum(n_sum - z_sum, 1.0)
    return (neg / pos).astype(np.float32)


def train_kof_detector(backbone: nn.Module, kof: nn.Module, feats_loader, device, pos_weight_vec: np.ndarray):
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    kof.to(device)

    posw = torch.from_numpy(pos_weight_vec).to(device)
    bce = nn.BCEWithLogitsLoss(reduction="none", pos_weight=posw)
    opt = torch.optim.AdamW(kof.parameters(), lr=args.LR_DETECT)

    tau_arr = np.array(args.TAU_LIST, dtype=float)
    ti = int(np.argmin(np.abs(tau_arr - float(args.TAU_TARGET))))
    head_w = np.ones(len(args.TAU_LIST), dtype=np.float32)
    head_w[ti] = 2.5
    head_w_t = torch.from_numpy(head_w).to(device).view(1, -1)

    for ep in range(1, int(args.EPOCHS_DETECT) + 1):
        kof.train()
        losses = []
        for feats, z_multi in tqdm(feats_loader, desc=f"[KOF] ep {ep}/{args.EPOCHS_DETECT}", leave=False):
            feats = feats.to(device)
            z_multi = z_multi.to(device).float()
            qlog = kof(feats)
            lv = bce(qlog, z_multi)
            loss = (lv * head_w_t).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))
        print(f"[KOF] ep={ep:02d} loss={np.mean(losses):.6f}")


# -----------------------------
# Fair allocation
# -----------------------------
def _k_from_budget(budget: float, n: int) -> int:
    budget = float(np.clip(budget, 0.0, 1.0))
    return int(np.floor(budget * float(n)))


def _allocate_parity_quotas(g_h: np.ndarray, budget: float) -> Dict[int, int]:
    uniq, counts = np.unique(g_h, return_counts=True)
    n_total = int(counts.sum())
    if n_total <= 0:
        return {}
    K = _k_from_budget(budget, n_total)
    if K <= 0:
        return {int(g): 0 for g in uniq}

    exact = (float(K) * counts.astype(float) / float(n_total))
    base = np.floor(exact).astype(int)
    frac = exact - base
    base = np.clip(base, 0, counts)

    R = K - int(base.sum())
    if R > 0:
        order = np.argsort(-frac)
        for i in order[:R]:
            if base[i] < counts[i]:
                base[i] += 1
    elif R < 0:
        order = np.argsort(frac)
        need = -R
        for i in order:
            if need == 0:
                break
            if base[i] > 0:
                base[i] -= 1
                need -= 1
    return {int(g): int(k) for g, k in zip(uniq, base)}


def _select_alerts_global(r_h: np.ndarray, budget: float) -> np.ndarray:
    n = len(r_h)
    if n == 0:
        return np.zeros(0, dtype=bool)
    K = _k_from_budget(budget, n)
    m = np.zeros(n, dtype=bool)
    if K <= 0:
        return m
    order = np.argsort(-r_h)
    m[order[:K]] = True
    return m


def _select_alerts_parity(r_h: np.ndarray, g_h: np.ndarray, budget: float) -> np.ndarray:
    n = len(r_h)
    if n == 0:
        return np.zeros(0, dtype=bool)
    quotas = _allocate_parity_quotas(g_h, budget)
    m = np.zeros(n, dtype=bool)
    for gg, k in quotas.items():
        idx = np.where(g_h == gg)[0]
        if len(idx) == 0 or k <= 0:
            continue
        order = idx[np.argsort(-r_h[idx])]
        m[order[:k]] = True
    return m


def _select_alerts_bounded_topk(r_h: np.ndarray, g_h: np.ndarray, budget: float, delta: float) -> np.ndarray:
    n = len(r_h)
    if n == 0:
        return np.zeros(n, dtype=bool)

    uniq, counts = np.unique(g_h, return_counts=True)
    n_total = int(counts.sum())
    K = _k_from_budget(budget, n_total)
    if K <= 0:
        return np.zeros(n, dtype=bool)

    delta = float(max(0.0, delta))
    bmin = max(0.0, float(budget) - delta)
    bmax = min(1.0, float(budget) + delta)

    Kmin = np.floor(bmin * counts.astype(float)).astype(int)
    Kmax = np.ceil(bmax * counts.astype(float)).astype(int)

    Kmin = np.clip(Kmin, 0, counts)
    Kmax = np.clip(Kmax, Kmin, counts)

    m = np.zeros(n, dtype=bool)
    picked = {int(g): 0 for g in uniq}

    # Satisfy lower bounds first.
    for gg, kmin in zip(uniq, Kmin):
        gg = int(gg)
        if int(kmin) <= 0:
            continue
        idx = np.where(g_h == gg)[0]
        order = idx[np.argsort(-r_h[idx])]
        take = int(min(int(kmin), len(order)))
        if take > 0:
            m[order[:take]] = True
            picked[gg] += take

    # Fill remaining by global score while respecting upper bounds.
    remaining = K - int(m.sum())
    if remaining > 0:
        global_order = np.argsort(-r_h)
        uniq_arr = np.asarray(uniq)
        for i in global_order:
            if remaining == 0:
                break
            if m[i]:
                continue
            gg = int(g_h[i])
            gi = int(np.where(uniq_arr == gg)[0][0])
            if picked[gg] < int(Kmax[gi]):
                m[i] = True
                picked[gg] += 1
                remaining -= 1

    # Final fallback if upper bounds prevent exact K.
    if remaining > 0:
        global_order = np.argsort(-r_h)
        for i in global_order:
            if remaining == 0:
                break
            if not m[i]:
                m[i] = True
                remaining -= 1

    return m


def _policy_alerts(r_h: np.ndarray, g_h: np.ndarray, budget: float, policy: str, delta: float) -> np.ndarray:
    if policy == "global":
        return _select_alerts_global(r_h, budget)
    if policy == "parity":
        return _select_alerts_parity(r_h, g_h, budget)
    if policy == "bounded":
        return _select_alerts_bounded_topk(r_h, g_h, budget, delta)
    raise ValueError(f"Unknown policy {policy}")


# -----------------------------
# Evaluation
# -----------------------------
def eval_monitoring_metrics(conf, err, risk_score, group, budgets, high_conf_region, policy="global", delta=0.0, tau_target=0.90, return_alerted=False):
    conf = np.asarray(conf).reshape(-1).astype(float)
    err = np.asarray(err).astype(int).reshape(-1)
    risk = np.asarray(risk_score).reshape(-1).astype(float)
    group = np.asarray(group).astype(int).reshape(-1)

    high = conf >= float(high_conf_region)
    high_tau = conf >= float(tau_target)

    hc_auc_tau = safe_auc(err[high_tau], risk[high_tau]) if high_tau.sum() > 0 else float("nan")
    if high.sum() == 0:
        out = {"hc_auc": float("nan"), "hc_auc_tau": float(hc_auc_tau), "alerts": [], "fairness": [], "groups": []}
        if return_alerted:
            out["alerted_masks"] = {}
        return out

    z = overconfident_error_label(err, conf, tau=float(tau_target))
    z_h, r_h, g_h, conf_h = z[high], risk[high], group[high], conf[high]
    hc_auc = safe_auc(z_h, r_h)

    denom = int(z_h.sum())
    n_high = int(len(z_h))
    z_rate_high = float(denom / n_high) if n_high > 0 else float("nan")

    alerts, fairness, group_rows = [], [], []
    alerted_masks = {}
    delta_out = float(delta) if policy == "bounded" else -1.0

    for b in budgets:
        b = float(b)
        alerted = _policy_alerts(r_h, g_h, b, policy=policy, delta=delta)
        if return_alerted:
            alerted_masks[b] = alerted.copy()

        n_alert = int(alerted.sum())
        tau_mask = conf_h >= float(tau_target)
        n_alert_tau = int((alerted & tau_mask).sum()) if n_alert > 0 else 0
        n_alert_mid = int((alerted & (~tau_mask)).sum()) if n_alert > 0 else 0
        frac_alert_tau = float(n_alert_tau / n_alert) if n_alert > 0 else float("nan")
        frac_alert_mid = float(n_alert_mid / n_alert) if n_alert > 0 else float("nan")

        prec = float(z_h[alerted].mean()) if n_alert > 0 else float("nan")
        rec = float(z_h[alerted].sum() / denom) if denom > 0 else (0.0 if n_alert == 0 else float("nan"))
        lift = float(prec / z_rate_high) if np.isfinite(prec) and np.isfinite(z_rate_high) and z_rate_high > 0 else float("nan")

        rates, gp, gr, gf = [], [], [], []
        n_groups_total = 0
        n_groups_prec_def = 0
        n_groups_rec_def = 0
        n_groups_fpr_def = 0

        for gg in np.unique(g_h):
            m = g_h == gg
            n_high_g = int(m.sum())
            if n_high_g == 0:
                continue
            n_groups_total += 1

            n_alert_g = int((alerted & m).sum())
            n_z_g = int(z_h[m].sum())
            n_alert_and_z_g = int(z_h[alerted & m].sum())

            ar = float(n_alert_g / n_high_g) if n_high_g > 0 else float("nan")
            if n_alert_g > 0:
                pg = float(n_alert_and_z_g / n_alert_g)
                n_groups_prec_def += 1
            else:
                pg = float("nan")
            if n_z_g > 0:
                rg = float(n_alert_and_z_g / n_z_g)
                n_groups_rec_def += 1
            else:
                rg = float("nan")

            n_neg_g = int(n_high_g - n_z_g)
            n_alert_and_negz_g = int(n_alert_g - n_alert_and_z_g)
            if n_neg_g > 0:
                fprg = float(n_alert_and_negz_g / n_neg_g)
                n_groups_fpr_def += 1
            else:
                fprg = float("nan")

            rates.append(ar)
            gp.append(pg)
            gr.append(rg)
            gf.append(fprg)
            group_rows.append({
                "budget": b, "policy": policy, "delta": delta_out, "group": int(gg),
                "n_high_g": n_high_g, "n_alert_g": n_alert_g, "n_z_g": n_z_g,
                "n_alert_and_z_g": n_alert_and_z_g, "n_neg_g": n_neg_g,
                "n_alert_and_negz_g": n_alert_and_negz_g,
                "alert_rate_g": ar, "precision_g": pg, "recall_g": rg, "fpr_g": fprg
            })

        if len(rates) >= 2 and np.isfinite(rates).any():
            rmin, rmax = float(np.nanmin(rates)), float(np.nanmax(rates))
            gap_ar = float(rmax - rmin)
            ratio_ar = float(rmax / rmin) if np.isfinite(rmin) and rmin > 0 else float("nan")
        else:
            gap_ar, ratio_ar = float("nan"), float("nan")

        gap_p = float(np.nanmax(gp) - np.nanmin(gp)) if np.isfinite(gp).any() else float("nan")
        gap_r = float(np.nanmax(gr) - np.nanmin(gr)) if np.isfinite(gr).any() else float("nan")
        gap_fpr = float(np.nanmax(gf) - np.nanmin(gf)) if np.isfinite(gf).any() else float("nan")
        worst_p = float(np.nanmin(gp)) if np.isfinite(gp).any() else float("nan")
        worst_r = float(np.nanmin(gr)) if np.isfinite(gr).any() else float("nan")
        worst_fpr = float(np.nanmax(gf)) if np.isfinite(gf).any() else float("nan")
        # Equal opportunity: TPR (recall) gap across groups.
        equal_opportunity_diff = gap_r
        # Equalized odds: worst of TPR-gap and FPR-gap across groups.
        equalized_odds_diff = (
            float(np.nanmax([gap_r, gap_fpr]))
            if (np.isfinite(gap_r) or np.isfinite(gap_fpr)) else float("nan")
        )

        alerts.append({
            "budget": b, "policy": policy, "delta": delta_out,
            "precision": prec, "recall": rec, "lift": lift,
            "z_rate_high": z_rate_high, "z_count_high": int(denom),
            "worst_precision": worst_p, "worst_recall": worst_r, "worst_fpr": worst_fpr,
            "n_alert": n_alert, "n_high": n_high,
            "n_groups_total": int(n_groups_total),
            "n_groups_precision_defined": int(n_groups_prec_def),
            "n_groups_recall_defined": int(n_groups_rec_def),
            "n_groups_fpr_defined": int(n_groups_fpr_def),
            "n_alert_tau": int(n_alert_tau), "n_alert_mid": int(n_alert_mid),
            "frac_alert_tau": float(frac_alert_tau), "frac_alert_mid": float(frac_alert_mid),
        })

        fairness.append({
            "budget": b, "policy": policy, "delta": delta_out,
            "alert_rate_gap": gap_ar, "alert_rate_ratio": ratio_ar,
            "gap_precision": gap_p, "gap_recall": gap_r,
            "gap_fpr": gap_fpr, "equal_opportunity_diff": equal_opportunity_diff,
            "equalized_odds_diff": equalized_odds_diff
        })

    out = {"hc_auc": float(hc_auc), "hc_auc_tau": float(hc_auc_tau), "alerts": alerts, "fairness": fairness, "groups": group_rows}
    if return_alerted:
        out["alerted_masks"] = alerted_masks
    return out


# -----------------------------
# NEW: bounded delta tuning for tcp_m_gate_tau
# -----------------------------
def _finite_or(x, fallback):
    x = float(x)
    return x if np.isfinite(x) else float(fallback)


def choose_best_bounded_delta(
    conf,
    err,
    risk_score,
    group,
    budgets,
    delta_grid,
    high_conf_region,
    tau_target=0.90,
):
    conf = np.asarray(conf).reshape(-1).astype(float)
    err = np.asarray(err).astype(int).reshape(-1)
    risk_score = np.asarray(risk_score).reshape(-1).astype(float)
    group = np.asarray(group).reshape(-1).astype(int)

    delta_grid = sorted({float(d) for d in delta_grid if float(d) >= 0.0})
    if len(delta_grid) == 0:
        delta_grid = [0.0]

    chosen = {}
    diag_rows = []

    for b in budgets:
        b = float(b)

        base = eval_monitoring_metrics(
            conf,
            err,
            risk_score,
            group,
            budgets=[b],
            high_conf_region=high_conf_region,
            policy="global",
            delta=0.0,
            tau_target=tau_target,
            return_alerted=False,
        )

        if len(base["alerts"]) == 0:
            chosen[b] = float(delta_grid[0])
            diag_rows.append({
                "budget": b,
                "chosen_delta": float(delta_grid[0]),
                "base_precision": float("nan"),
                "base_recall": float("nan"),
                "base_lift": float("nan"),
                "base_gap_precision": float("nan"),
                "base_gap_recall": float("nan"),
                "chosen_precision": float("nan"),
                "chosen_recall": float("nan"),
                "chosen_lift": float("nan"),
                "chosen_gap_precision": float("nan"),
                "chosen_gap_recall": float("nan"),
            })
            continue

        a0 = base["alerts"][0]
        f0 = base["fairness"][0]

        base_precision = _finite_or(a0["precision"], 0.0)
        base_recall = _finite_or(a0["recall"], 0.0)
        base_lift = _finite_or(a0["lift"], 0.0)

        prec_floor = 0.95 * base_precision
        rec_floor = 0.95 * base_recall
        lift_floor = 0.90 * base_lift

        best = None

        for d in delta_grid:
            rr = eval_monitoring_metrics(
                conf,
                err,
                risk_score,
                group,
                budgets=[b],
                high_conf_region=high_conf_region,
                policy="bounded",
                delta=float(d),
                tau_target=tau_target,
                return_alerted=False,
            )

            if len(rr["alerts"]) == 0:
                continue

            a1 = rr["alerts"][0]
            f1 = rr["fairness"][0]

            p1 = _finite_or(a1["precision"], -1e9)
            r1 = _finite_or(a1["recall"], -1e9)
            l1 = _finite_or(a1["lift"], -1e9)

            gp1 = _finite_or(f1["gap_precision"], 1e6)
            gr1 = _finite_or(f1["gap_recall"], 1e6)
            fairness_obj = gp1 + gr1

            prec_violation = max(0.0, prec_floor - p1)
            rec_violation = max(0.0, rec_floor - r1)
            lift_violation = max(0.0, lift_floor - l1)

            feasible = (prec_violation <= 0.0) and (rec_violation <= 0.0) and (lift_violation <= 0.0)

            key = (
                0 if feasible else 1,
                gr1,                # EO as primary fairness metric
                fairness_obj,              # fallback: gap_precision + gap_recall
                1000.0 * rec_violation + 700.0 * prec_violation + 400.0 * lift_violation,
                float(d), -r1, -p1, -l1
            )

            cand = {
                "key": key,
                "delta": float(d),
                "precision": float(a1["precision"]),
                "recall": float(a1["recall"]),
                "lift": float(a1["lift"]),
                "gap_precision": float(f1["gap_precision"]),
                "gap_recall": float(f1["gap_recall"]),
            }

            if (best is None) or (cand["key"] < best["key"]):
                best = cand

        if best is None:
            chosen[b] = float(delta_grid[0])
            diag_rows.append({
                "budget": b,
                "chosen_delta": float(delta_grid[0]),
                "base_precision": float(a0["precision"]),
                "base_recall": float(a0["recall"]),
                "base_lift": float(a0["lift"]),
                "base_gap_precision": float(f0["gap_precision"]),
                "base_gap_recall": float(f0["gap_recall"]),
                "chosen_precision": float("nan"),
                "chosen_recall": float("nan"),
                "chosen_lift": float("nan"),
                "chosen_gap_precision": float("nan"),
                "chosen_gap_recall": float("nan"),
            })
        else:
            chosen[b] = float(best["delta"])
            diag_rows.append({
                "budget": b,
                "chosen_delta": float(best["delta"]),
                "base_precision": float(a0["precision"]),
                "base_recall": float(a0["recall"]),
                "base_lift": float(a0["lift"]),
                "base_gap_precision": float(f0["gap_precision"]),
                "base_gap_recall": float(f0["gap_recall"]),
                "chosen_precision": float(best["precision"]),
                "chosen_recall": float(best["recall"]),
                "chosen_lift": float(best["lift"]),
                "chosen_gap_precision": float(best["gap_precision"]),
                "chosen_gap_recall": float(best["gap_recall"]),
            })

    return chosen, pd.DataFrame(diag_rows)


def eval_monitoring_metrics_budgetwise_bounded(
    conf,
    err,
    risk_score,
    group,
    budget_to_delta,
    high_conf_region,
    tau_target=0.90,
    policy_name="bounded_tuned",
    return_alerted=False,
):
    conf = np.asarray(conf).reshape(-1).astype(float)
    err = np.asarray(err).astype(int).reshape(-1)
    risk = np.asarray(risk_score).reshape(-1).astype(float)
    group = np.asarray(group).astype(int).reshape(-1)

    budgets = sorted([float(b) for b in budget_to_delta.keys()])

    high = conf >= float(high_conf_region)
    high_tau = conf >= float(tau_target)

    hc_auc_tau = safe_auc(err[high_tau], risk[high_tau]) if high_tau.sum() > 0 else float("nan")
    if high.sum() == 0:
        out = {"hc_auc": float("nan"), "hc_auc_tau": float(hc_auc_tau), "alerts": [], "fairness": [], "groups": []}
        if return_alerted:
            out["alerted_masks"] = {}
        return out

    z = overconfident_error_label(err, conf, tau=float(tau_target))
    z_h, r_h, g_h, conf_h = z[high], risk[high], group[high], conf[high]
    hc_auc = safe_auc(z_h, r_h)

    denom = int(z_h.sum())
    n_high = int(len(z_h))
    z_rate_high = float(denom / n_high) if n_high > 0 else float("nan")

    alerts, fairness, group_rows = [], [], []
    alerted_masks = {}

    for b in budgets:
        b = float(b)
        d = float(budget_to_delta.get(b, 0.0))
        alerted = _policy_alerts(r_h, g_h, b, policy="bounded", delta=d)
        if return_alerted:
            alerted_masks[b] = alerted.copy()

        n_alert = int(alerted.sum())
        tau_mask = conf_h >= float(tau_target)
        n_alert_tau = int((alerted & tau_mask).sum()) if n_alert > 0 else 0
        n_alert_mid = int((alerted & (~tau_mask)).sum()) if n_alert > 0 else 0
        frac_alert_tau = float(n_alert_tau / n_alert) if n_alert > 0 else float("nan")
        frac_alert_mid = float(n_alert_mid / n_alert) if n_alert > 0 else float("nan")

        prec = float(z_h[alerted].mean()) if n_alert > 0 else float("nan")
        rec = float(z_h[alerted].sum() / denom) if denom > 0 else (0.0 if n_alert == 0 else float("nan"))
        lift = float(prec / z_rate_high) if np.isfinite(prec) and np.isfinite(z_rate_high) and z_rate_high > 0 else float("nan")

        rates, gp, gr, gf = [], [], [], []
        n_groups_total = 0
        n_groups_prec_def = 0
        n_groups_rec_def = 0
        n_groups_fpr_def = 0

        for gg in np.unique(g_h):
            m = g_h == gg
            n_high_g = int(m.sum())
            if n_high_g == 0:
                continue
            n_groups_total += 1

            n_alert_g = int((alerted & m).sum())
            n_z_g = int(z_h[m].sum())
            n_alert_and_z_g = int(z_h[alerted & m].sum())

            ar = float(n_alert_g / n_high_g) if n_high_g > 0 else float("nan")
            if n_alert_g > 0:
                pg = float(n_alert_and_z_g / n_alert_g)
                n_groups_prec_def += 1
            else:
                pg = float("nan")
            if n_z_g > 0:
                rg = float(n_alert_and_z_g / n_z_g)
                n_groups_rec_def += 1
            else:
                rg = float("nan")

            n_neg_g = int(n_high_g - n_z_g)
            n_alert_and_negz_g = int(n_alert_g - n_alert_and_z_g)
            if n_neg_g > 0:
                fprg = float(n_alert_and_negz_g / n_neg_g)
                n_groups_fpr_def += 1
            else:
                fprg = float("nan")

            rates.append(ar)
            gp.append(pg)
            gr.append(rg)
            gf.append(fprg)
            group_rows.append({
                "budget": b, "policy": policy_name, "delta": d, "group": int(gg),
                "n_high_g": n_high_g, "n_alert_g": n_alert_g, "n_z_g": n_z_g,
                "n_alert_and_z_g": n_alert_and_z_g, "n_neg_g": n_neg_g,
                "n_alert_and_negz_g": n_alert_and_negz_g,
                "alert_rate_g": ar, "precision_g": pg, "recall_g": rg, "fpr_g": fprg
            })

        if len(rates) >= 2 and np.isfinite(rates).any():
            rmin, rmax = float(np.nanmin(rates)), float(np.nanmax(rates))
            gap_ar = float(rmax - rmin)
            ratio_ar = float(rmax / rmin) if np.isfinite(rmin) and rmin > 0 else float("nan")
        else:
            gap_ar, ratio_ar = float("nan"), float("nan")

        gap_p = float(np.nanmax(gp) - np.nanmin(gp)) if np.isfinite(gp).any() else float("nan")
        gap_r = float(np.nanmax(gr) - np.nanmin(gr)) if np.isfinite(gr).any() else float("nan")
        gap_fpr = float(np.nanmax(gf) - np.nanmin(gf)) if np.isfinite(gf).any() else float("nan")
        worst_p = float(np.nanmin(gp)) if np.isfinite(gp).any() else float("nan")
        worst_r = float(np.nanmin(gr)) if np.isfinite(gr).any() else float("nan")
        worst_fpr = float(np.nanmax(gf)) if np.isfinite(gf).any() else float("nan")
        # Equal opportunity: TPR (recall) gap across groups.
        equal_opportunity_diff = gap_r
        # Equalized odds: worst of TPR-gap and FPR-gap across groups.
        equalized_odds_diff = (
            float(np.nanmax([gap_r, gap_fpr]))
            if (np.isfinite(gap_r) or np.isfinite(gap_fpr)) else float("nan")
        )

        alerts.append({
            "budget": b, "policy": policy_name, "delta": d,
            "precision": prec, "recall": rec, "lift": lift,
            "z_rate_high": z_rate_high, "z_count_high": int(denom),
            "worst_precision": worst_p, "worst_recall": worst_r,
            "n_alert": n_alert, "n_high": n_high, "n_groups_total": int(n_groups_total),
            "n_groups_precision_defined": int(n_groups_prec_def),
            "n_groups_recall_defined": int(n_groups_rec_def),
            "n_alert_tau": int(n_alert_tau), "n_alert_mid": int(n_alert_mid),
            "frac_alert_tau": float(frac_alert_tau), "frac_alert_mid": float(frac_alert_mid),
        })

        fairness.append({
            "budget": b, "policy": policy_name, "delta": d,
            "alert_rate_gap": gap_ar, "alert_rate_ratio": ratio_ar,
            "gap_precision": gap_p, "gap_recall": gap_r,
            "gap_fpr": gap_fpr, "equal_opportunity_diff": equal_opportunity_diff,
            "equalized_odds_diff": equalized_odds_diff
        })

    out = {"hc_auc": float(hc_auc), "hc_auc_tau": float(hc_auc_tau), "alerts": alerts, "fairness": fairness, "groups": group_rows}
    if return_alerted:
        out["alerted_masks"] = alerted_masks
    return out


# -----------------------------
# Plot helper
# -----------------------------
def plot_line_ci(df, xcol, meancol, locol, hicol, huecol, title, ylabel, outpath, order=None):
    plt.figure()
    groups = order or df[huecol].unique().tolist()
    for g in groups:
        d = df[df[huecol] == g].sort_values(xcol)
        if len(d) == 0:
            continue
        x = d[xcol].to_numpy()
        m = d[meancol].to_numpy()
        lo = d[locol].to_numpy()
        hi = d[hicol].to_numpy()
        plt.plot(x, m, marker="o", label=str(g))
        if np.isfinite(lo).any() and np.isfinite(hi).any():
            plt.fill_between(x, lo, hi, alpha=0.15)
    plt.xlabel(xcol)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=160)
    plt.close()

# -----------------------------
# One seed / experiment run
# -----------------------------
def run_one_seed_experiment(seed: int, cfg: ExperimentConfig):
    set_seed(seed)
    set_determinism(args.DETERMINISTIC)
    device = get_device(args.DEVICE)
    pin = torch.cuda.is_available()

    print("\n" + "=" * 80)
    print(f"RUN seed={seed} exp={cfg.name} device={device}")
    print("=" * 80)

    all_states = args.TRAIN_STATES + args.SHIFT_STATES
    year_data = {yr: load_acs_income_year(yr, all_states, args.DATA_DIR, args.MAX_PER_STATE_YEAR, seed) for yr in args.YEARS}

    train_years = [2014, 2015, 2016]
    train_parts = [filter_by_states(year_data[yr], args.TRAIN_STATES) for yr in train_years]
    X_train = pd.concat([p["X"] for p in train_parts], axis=0).reset_index(drop=True)
    y_train = np.concatenate([p["y"] for p in train_parts]).astype(np.int64).reshape(-1)
    g_train = np.concatenate([p["g"] for p in train_parts]).astype(np.int64).reshape(-1)

    d2017_train = filter_by_states(year_data[2017], args.TRAIN_STATES)
    d2017_shift = filter_by_states(year_data[2017], args.SHIFT_STATES)

    idx = np.arange(len(d2017_train["y"]))
    np.random.default_rng(seed + 2017).shuffle(idx)
    n_vs = int(0.5 * len(idx))
    val_select = slice_by_idx(d2017_train, idx[:n_vs])
    val_calib = slice_by_idx(d2017_train, idx[n_vs:])

    swap_col = pick_existing_col(args.SWAP_COL_PREFS, X_train)
    perm_col = pick_existing_col(args.PERM_COL_PREFS, X_train, exclude={swap_col})
    swap_map = build_swap_mapping_label_informed(X_train, y_train, swap_col, args.SWAP_TOPK)
    perm_map = build_perm_mapping_label_free(X_train[perm_col], args.PERM_TOPK, seed)

    cat_cols, cont_cols = infer_cat_cont_cols(X_train, force_cat_cols=[swap_col, perm_col])
    cat_maps, cat_cards, cont_mean, cont_std = fit_encoders(X_train, cat_cols, cont_cols)

    Xc_tr, Xn_tr, y_tr = encode_split(X_train, y_train, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    Xc_vs, Xn_vs, y_vs = encode_split(val_select["X"], val_select["y"], cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
    Xc_vc, Xn_vc, y_vc = encode_split(val_calib["X"], val_calib["y"], cat_cols, cont_cols, cat_maps, cont_mean, cont_std)

    ds_tr = TabDataset(Xc_tr, Xn_tr, y_tr)
    ds_vs = TabDataset(Xc_vs, Xn_vs, y_vs)
    ds_vc = TabDataset(Xc_vc, Xn_vc, y_vc)

    train_loader = DataLoader(ds_tr, batch_size=args.BATCH_SIZE, shuffle=True, num_workers=args.NUM_WORKERS, pin_memory=pin)
    val_loader = DataLoader(ds_vs, batch_size=args.BATCH_SIZE, shuffle=False, num_workers=args.NUM_WORKERS, pin_memory=pin)
    calib_loader = DataLoader(ds_vc, batch_size=args.BATCH_SIZE, shuffle=False, num_workers=args.NUM_WORKERS, pin_memory=pin)

    clf = TabMLP(cat_cards=cat_cards, n_cont=len(cont_cols), hidden=args.HIDDEN, dropout=args.DROPOUT)
    clf = train_cls_with_early_stopping(clf, train_loader, val_loader, device)
    temp_scaler = fit_temperature_scaler(clf, calib_loader, device) if args.USE_TEMP_SCALING else None

    @torch.no_grad()
    def collect_split_outputs(X_df, y_arr):
        Xc, Xn, yy = encode_split(X_df, y_arr, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
        loader = DataLoader(TabDataset(Xc, Xn, yy), batch_size=args.BATCH_SIZE, shuffle=False, num_workers=0)
        clf.eval()
        if temp_scaler is not None:
            temp_scaler.eval()

        feats_all, logits_raw_all, logits_cal_all = [], [], []
        conf_all, pred_all, err_all, ent_all, marg_all = [], [], [], [], []

        for Xc_b, Xn_b, y_b in loader:
            Xc_b, Xn_b, y_b = Xc_b.to(device), Xn_b.to(device), y_b.to(device)
            feats_b, logits_raw_b, logits_cal_b, conf_b, marg_b, ent_b = make_feat_batch(clf, Xc_b, Xn_b, temp_scaler=temp_scaler)
            pred_b = logits_cal_b.argmax(dim=-1)
            err_b = (pred_b != y_b).int()

            feats_all.append(feats_b.cpu())
            logits_raw_all.append(logits_raw_b.cpu())
            logits_cal_all.append(logits_cal_b.cpu())
            conf_all.append(conf_b.cpu())
            pred_all.append(pred_b.cpu())
            err_all.append(err_b.cpu())
            ent_all.append(ent_b.cpu())
            marg_all.append(marg_b.cpu())

        feats = torch.cat(feats_all, dim=0)
        logits_raw = torch.cat(logits_raw_all, dim=0)
        logits_cal = torch.cat(logits_cal_all, dim=0)
        conf = torch.cat(conf_all, dim=0).numpy()
        pred = torch.cat(pred_all, dim=0).numpy().astype(int)
        err = torch.cat(err_all, dim=0).numpy().astype(int)
        ent = torch.cat(ent_all, dim=0).numpy()
        marg = torch.cat(marg_all, dim=0).numpy()
        return feats, logits_raw, logits_cal, conf, pred, err, ent, marg

    feats_vs, logits_raw_vs, logits_cal_vs, vs_conf, vs_pred, vs_err, vs_ent, vs_marg = collect_split_outputs(val_select["X"], val_select["y"])
    feats_vc, logits_raw_vc, logits_cal_vc, vc_conf, vc_pred, vc_err, vc_ent, vc_marg = collect_split_outputs(val_calib["X"], val_calib["y"])

    z_vs = overconfident_error_label(vs_err, vs_conf, tau=args.TAU_TARGET).astype(np.int64)
    z_vc = overconfident_error_label(vc_err, vc_conf, tau=args.TAU_TARGET).astype(np.int64)

    # RED-like baseline
    red_lr = fit_red_like_lr(
        np.stack([vs_conf, vs_ent, vs_marg], axis=1).astype(np.float32),
        z_vs,
        seed=seed,
    )

    # TCP
    class TCPTrainDataset(Dataset):
        def __init__(self, feats, y_true, pred):
            self.feats = feats.float()
            self.y = torch.from_numpy(np.asarray(y_true).astype(np.int64)).long()
            self.pred = torch.from_numpy(np.asarray(pred).astype(np.int64)).long()

        def __len__(self):
            return int(self.y.numel())

        def __getitem__(self, i):
            return self.feats[i], self.y[i], self.pred[i]

    tcp = TCPHead(in_dim=feats_vs.shape[1], hidden=args.HIDDEN)
    tcp_loader = DataLoader(
        TCPTrainDataset(feats_vs, val_select["y"], vs_pred),
        batch_size=1024,
        shuffle=True,
        num_workers=0,
    )
    train_tcp_correctness(clf, tcp, tcp_loader, device)

    # TCP-M
    tcpm = TCPHead(in_dim=feats_vs.shape[1], hidden=args.HIDDEN)

    if args.TCPM_TRAIN_ON_TAU_SLICE:
        tcpm_keep = vs_conf >= float(args.TAU_TARGET)
    else:
        tcpm_keep = np.ones(len(vs_conf), dtype=bool)

    feats_tcpm = feats_vs[tcpm_keep]
    z_tcpm = torch.from_numpy(z_vs[tcpm_keep].astype(np.float32))

    train_tcp_monitoring_strong(
        clf, tcpm,
        feats_train=feats_tcpm, z_train=z_tcpm, device=device,
        val_feats=feats_vs, val_conf=vs_conf, val_z=z_vs
    )

    # NEW: choose bounded delta by budget on validation for tcp_m_gate_tau
    with torch.no_grad():
        tcpm_val_logit = tcpm(feats_vs.to(device)).cpu().numpy().reshape(-1)
    risk_tcpm_val = _sigmoid_np(tcpm_val_logit).astype(np.float32)
    if args.TCPM_APPLY_TAU_GATE_AT_INFERENCE:
        risk_tcpm_val[vs_conf < float(args.TAU_TARGET)] = -1e9

    tcpm_bounded_delta_by_budget, tcpm_bounded_delta_diag = choose_best_bounded_delta(
        conf=vs_conf,
        err=vs_err,
        risk_score=risk_tcpm_val,
        group=np.asarray(val_select["g"]).reshape(-1).astype(np.int64),
        budgets=args.ALERT_BUDGETS,
        delta_grid=args.FAIR_TOPK_DELTAS,
        high_conf_region=args.HIGH_CONF_REGION,
        tau_target=args.TAU_TARGET,
    )
    print(f"[TCP-M gate tau | bounded tuned] chosen delta by budget = {tcpm_bounded_delta_by_budget}")

    # KOF without stability
    z_multi_vs = make_z_multi_tau(
        err=torch.from_numpy(vs_err.astype(np.float32)),
        conf=torch.from_numpy(vs_conf.astype(np.float32)),
        tau_list=args.TAU_LIST,
    ).float()

    class KOFTrainDataset(Dataset):
        def __init__(self, feats, z_multi):
            self.feats = feats.float()
            self.z_multi = z_multi.float()

        def __len__(self):
            return int(self.z_multi.shape[0])

        def __getitem__(self, i):
            return self.feats[i], self.z_multi[i]

    kof_nostab = KOFDetector(in_dim=feats_vs.shape[1], hidden=args.KOF_HIDDEN, n_tau=len(args.TAU_LIST))
    posw_vec = compute_kof_pos_weight(torch.from_numpy(y_vs), logits_cal_vs)
    kof_loader_nostab = DataLoader(KOFTrainDataset(feats_vs, z_multi_vs), batch_size=1024, shuffle=True, num_workers=0)
    train_kof_detector(clf, kof_nostab, kof_loader_nostab, device, posw_vec)

    with torch.no_grad():
        lg_vs = kof_nostab(feats_vs.to(device)).cpu().numpy()
        lg_vc = kof_nostab(feats_vc.to(device)).cpu().numpy()
        ph_vs = kof_head_probs(lg_vs)
        ph_vc = kof_head_probs(lg_vc)

    combo_lr_nostab = fit_tau_combo_lr(ph_vs.astype(np.float32), z_vs, seed=seed)
    ti = int(np.argmin(np.abs(np.asarray(args.TAU_LIST) - float(args.TAU_TARGET))))
    risk_candidates_vs_nostab = {
        "head": ph_vs[:, ti],
        "mean": ph_vs.mean(axis=1),
        "combo": combo_lr_nostab.predict_proba(ph_vs.astype(np.float32))[:, 1],
    }
    aucs_nostab = {k: safe_auc(z_vs[vs_conf >= args.HIGH_CONF_REGION], v[vs_conf >= args.HIGH_CONF_REGION]) for k, v in risk_candidates_vs_nostab.items()}
    best_nostab = max(aucs_nostab, key=lambda k: (-1 if not np.isfinite(aucs_nostab[k]) else aucs_nostab[k]))
    print(f"[KOF no-stab] best={best_nostab} AUCs={aucs_nostab}")

    risk_vc_nostab = {
        "head": ph_vc[:, ti],
        "mean": ph_vc.mean(axis=1),
        "combo": combo_lr_nostab.predict_proba(ph_vc.astype(np.float32))[:, 1],
    }[best_nostab]
    iso_nostab = calibrate_isotonic(risk_vc_nostab[vc_conf >= args.HIGH_CONF_REGION], z_vc[vc_conf >= args.HIGH_CONF_REGION])

    # KOF with stability
    kops = KnowledgeOps(cat_cols, cont_cols, X_train, seed)

    if args.USE_STABILITY:
        n_sub = min(len(val_select["X"]), int(args.STAB_MAX_SAMPLES_FOR_FEAT))
        sub_idx = np.arange(len(val_select["X"]))[:n_sub]

        stab_vs = compute_stability_features(
            clf, temp_scaler,
            val_select["X"].iloc[sub_idx].reset_index(drop=True),
            np.asarray(val_select["y"])[sub_idx],
            kops, cat_cols, cont_cols, cat_maps, cont_mean, cont_std,
            device, args.STAB_K, args.STAB_PERT_PROB, args.STAB_CONT_NOISE_STD, args.BATCH_SIZE
        )
        feats_vs_stab = torch.cat([feats_vs[sub_idx], torch.from_numpy(stab_vs).float()], dim=1)
        z_multi_vs_stab = z_multi_vs[sub_idx]

        n_sub_vc = min(len(val_calib["X"]), int(args.STAB_MAX_SAMPLES_FOR_FEAT))
        sub_idx_vc = np.arange(len(val_calib["X"]))[:n_sub_vc]
        stab_vc = compute_stability_features(
            clf, temp_scaler,
            val_calib["X"].iloc[sub_idx_vc].reset_index(drop=True),
            np.asarray(val_calib["y"])[sub_idx_vc],
            kops, cat_cols, cont_cols, cat_maps, cont_mean, cont_std,
            device, args.STAB_K, args.STAB_PERT_PROB, args.STAB_CONT_NOISE_STD, args.BATCH_SIZE
        )
        feats_vc_stab = torch.cat([feats_vc[sub_idx_vc], torch.from_numpy(stab_vc).float()], dim=1)

        kof_stab = KOFDetector(in_dim=feats_vs_stab.shape[1], hidden=args.KOF_HIDDEN, n_tau=len(args.TAU_LIST))
        posw_vec_stab = compute_kof_pos_weight(torch.from_numpy(y_vs[sub_idx]), logits_cal_vs[sub_idx])
        kof_loader_stab = DataLoader(KOFTrainDataset(feats_vs_stab, z_multi_vs_stab), batch_size=1024, shuffle=True, num_workers=0)
        train_kof_detector(clf, kof_stab, kof_loader_stab, device, posw_vec_stab)

        with torch.no_grad():
            lgS_vs = kof_stab(feats_vs_stab.to(device)).cpu().numpy()
            lgS_vc = kof_stab(feats_vc_stab.to(device)).cpu().numpy()
            phS_vs = kof_head_probs(lgS_vs)
            phS_vc = kof_head_probs(lgS_vc)

        combo_lr_stab = fit_tau_combo_lr(phS_vs.astype(np.float32), z_vs[sub_idx], seed=seed)
        risk_candidates_vs_stab = {
            "head": phS_vs[:, ti],
            "mean": phS_vs.mean(axis=1),
            "combo": combo_lr_stab.predict_proba(phS_vs.astype(np.float32))[:, 1],
        }
        aucs_stab = {k: safe_auc(z_vs[sub_idx][vs_conf[sub_idx] >= args.HIGH_CONF_REGION], v[vs_conf[sub_idx] >= args.HIGH_CONF_REGION]) for k, v in risk_candidates_vs_stab.items()}
        best_stab = max(aucs_stab, key=lambda k: (-1 if not np.isfinite(aucs_stab[k]) else aucs_stab[k]))
        print(f"[KOF stab] best={best_stab} AUCs={aucs_stab}")

        risk_vc_stab = {
            "head": phS_vc[:, ti],
            "mean": phS_vc.mean(axis=1),
            "combo": combo_lr_stab.predict_proba(phS_vc.astype(np.float32))[:, 1],
        }[best_stab]
        iso_stab = calibrate_isotonic(
            risk_vc_stab[vc_conf[sub_idx_vc] >= args.HIGH_CONF_REGION],
            z_vc[sub_idx_vc][vc_conf[sub_idx_vc] >= args.HIGH_CONF_REGION]
        )
    else:
        kof_stab = None
        combo_lr_stab = None
        best_stab = "combo"
        iso_stab = None

    def sample_target_mix(rho: float, rng: np.random.Generator):
        n_total = int(args.N_TARGET)
        n_shift = int(round(float(rho) * n_total))
        n_shift = int(np.clip(n_shift, 0, n_total))
        n_train = int(n_total - n_shift)

        idx_train = rng.choice(len(d2017_train["y"]), size=n_train, replace=(n_train > len(d2017_train["y"])))
        idx_shift = rng.choice(len(d2017_shift["y"]), size=n_shift, replace=(n_shift > len(d2017_shift["y"])))

        X_id = d2017_train["X"].iloc[idx_train].reset_index(drop=True)
        y_id = np.asarray(d2017_train["y"])[idx_train].reshape(-1)
        g_id = np.asarray(d2017_train["g"])[idx_train].reshape(-1)

        X_ood = d2017_shift["X"].iloc[idx_shift].reset_index(drop=True)
        y_ood = np.asarray(d2017_shift["y"])[idx_shift].reshape(-1)
        g_ood = np.asarray(d2017_shift["g"])[idx_shift].reshape(-1)

        X_te = pd.concat([X_id, X_ood], axis=0).reset_index(drop=True)
        y_te = np.concatenate([y_id, y_ood]).astype(np.int64)
        g_te = np.concatenate([g_id, g_ood]).astype(np.int64)

        if float(rho) > 0:
            p_drift = p_drift_from_rho(rho)

            if n_shift > 0:
                X_ood_drift, _ = apply_combined_drift(
                    cfg=cfg,
                    X_df=X_ood,
                    rho=rho,
                    seed=seed + int(1e6 * float(rho)) + 77,
                    swap_col=swap_col,
                    perm_col=perm_col,
                    swap_map=swap_map,
                    perm_map=perm_map,
                    p_override=p_drift,
                )
            else:
                X_ood_drift = X_ood

            if args.DRIFT_ON_NONSHIFT_AT_HIGH_RHO and n_train > 0 and float(rho) >= 0.8:
                X_id_drift, _ = apply_combined_drift(
                    cfg=cfg,
                    X_df=X_id,
                    rho=rho,
                    seed=seed + int(1e6 * float(rho)) + 99,
                    swap_col=swap_col,
                    perm_col=perm_col,
                    swap_map=swap_map,
                    perm_map=perm_map,
                    p_override=float(args.NONSHIFT_DRIFT_SCALE) * p_drift,
                )
            else:
                X_id_drift = X_id

            X_te = pd.concat([X_id_drift, X_ood_drift], axis=0).reset_index(drop=True)

        perm = rng.permutation(len(y_te))
        X_te = X_te.iloc[perm].reset_index(drop=True)
        y_te = y_te[perm]
        g_te = g_te[perm]
        return X_te, y_te, g_te

    @torch.no_grad()
    def score_batch(X_df, y):
        Xc, Xn, yy = encode_split(X_df, y, cat_cols, cont_cols, cat_maps, cont_mean, cont_std)
        loader = DataLoader(TabDataset(Xc, Xn, yy), batch_size=args.BATCH_SIZE, shuffle=False, num_workers=0)
        feats, logits_cal, conf, err, ent, marg, ener, pred = [], [], [], [], [], [], [], []
        clf.eval()
        if temp_scaler is not None:
            temp_scaler.eval()
        for Xc_b, Xn_b, y_b in loader:
            Xc_b, Xn_b, y_b = Xc_b.to(device), Xn_b.to(device), y_b.to(device)
            feats_b, _, lg_cal_b, cf, mg, en = make_feat_batch(clf, Xc_b, Xn_b, temp_scaler=temp_scaler)
            pr = lg_cal_b.argmax(dim=-1)
            er = (pr != y_b).int()
            with torch.no_grad():
                feats.append(feats_b.cpu())
                logits_cal.append(lg_cal_b.cpu())
                conf.append(cf.cpu())
                err.append(er.cpu())
                ent.append(en.cpu())
                marg.append(mg.cpu())
                ener.append(energy_from_logits(lg_cal_b).cpu())
                pred.append(pr.cpu())
        feats = torch.cat(feats, dim=0)
        logits_cal = torch.cat(logits_cal, dim=0)
        conf = torch.cat(conf, dim=0).numpy()
        err = torch.cat(err, dim=0).numpy().astype(int)
        ent = torch.cat(ent, dim=0).numpy()
        marg = torch.cat(marg, dim=0).numpy()
        ener = torch.cat(ener, dim=0).numpy()
        pred = torch.cat(pred, dim=0).numpy().astype(int)
        return feats, logits_cal, conf, err, pred, ent, marg, ener

    overall_rows, alert_rows, fair_rows, group_rows, selchg_rows, dataset_stats_rows = [], [], [], [], [], []

    # Base split statistics (train / validation-select / validation-calib), consistent with diabetes script.
    for _split_name, _y_sp, _g_sp in [
        ("train", y_train, g_train),
        ("val_select", val_select["y"], val_select["g"]),
        ("val_calib", val_calib["y"], val_calib["g"]),
    ]:
        _sstats = compute_split_stats(f"{cfg.name}/{_split_name}", None, _y_sp, _g_sp)
        _sstats["exp"] = cfg.name
        _sstats["seed"] = seed
        dataset_stats_rows.append(_sstats)

    for rho in args.RHO_VALUES:
        rng_e = np.random.default_rng(seed + 5000 + int(1e6 * rho))
        X_te_df, y_te, g_te = sample_target_mix(rho, rng_e)

        feats_te, logits_cal_te, conf_te, err_te, pred_te, ent_te, marg_te, energy_te = score_batch(X_te_df, y_te)

        # Dataset/shift statistics under this shift condition (consistent with diabetes script's KBS #2 fix).
        z_te_stats = overconfident_error_label(err_te, conf_te, args.TAU_TARGET)
        high_te_stats = (conf_te >= float(args.HIGH_CONF_REGION))
        split_stats = compute_split_stats(
            f"{cfg.name}/target_mix", float(rho), y_te, g_te,
            z=z_te_stats, high_mask=high_te_stats,
        )
        split_stats["exp"] = cfg.name
        split_stats["seed"] = seed
        dataset_stats_rows.append(split_stats)

        stab_te = None
        if args.USE_STABILITY and (kof_stab is not None):
            stab_te = compute_stability_features(
                clf, temp_scaler, X_te_df, y_te, kops, cat_cols, cont_cols, cat_maps, cont_mean, cont_std,
                device, args.STAB_K, args.STAB_PERT_PROB, args.STAB_CONT_NOISE_STD, args.BATCH_SIZE
            )

        risk_ent = ent_te
        risk_energy = energy_te
        risk_red = red_lr.predict_proba(np.stack([conf_te, ent_te, marg_te], axis=1).astype(np.float32))[:, 1]

        with torch.no_grad():
            tcp_logit = tcp(feats_te.to(device)).cpu().numpy().reshape(-1)
        risk_tcp = 1.0 - _sigmoid_np(tcp_logit)

        risk_tcp_gate_tau = risk_tcp.copy()
        risk_tcp_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        rng_rand = np.random.default_rng(seed + 90000 + int(1e6 * float(rho)))
        risk_rand_gate_tau = rng_rand.random(len(conf_te)).astype(np.float32)
        risk_rand_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        with torch.no_grad():
            tcpm_logit = tcpm(feats_te.to(device)).cpu().numpy().reshape(-1)
        risk_tcpm = _sigmoid_np(tcpm_logit).astype(np.float32)

        risk_tcpm_gate_tau = risk_tcpm.copy()
        risk_tcpm_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        with torch.no_grad():
            lg0 = kof_nostab(feats_te.to(device)).cpu().numpy()
            ph0 = kof_head_probs(lg0)

        risk_kof_nostab = {
            "head": ph0[:, ti],
            "mean": ph0.mean(axis=1),
            "combo": combo_lr_nostab.predict_proba(ph0.astype(np.float32))[:, 1],
        }[best_nostab]

        if iso_nostab is not None:
            high_te = conf_te >= float(args.HIGH_CONF_REGION)
            r = risk_kof_nostab.copy()
            r[high_te] = iso_nostab.predict(r[high_te])
            risk_kof_nostab = r

        if kof_stab is not None and stab_te is not None and combo_lr_stab is not None:
            with torch.no_grad():
                lgS = kof_stab(
                    torch.cat([feats_te, torch.from_numpy(stab_te).float()], dim=1).to(device)
                ).cpu().numpy()
                phS = kof_head_probs(lgS)

            risk_kof_stab = {
                "head": phS[:, ti],
                "mean": phS.mean(axis=1),
                "combo": combo_lr_stab.predict_proba(phS.astype(np.float32))[:, 1],
            }[best_stab]

            if iso_stab is not None:
                high_te = conf_te >= float(args.HIGH_CONF_REGION)
                r = risk_kof_stab.copy()
                r[high_te] = iso_stab.predict(r[high_te])
                risk_kof_stab = r
        else:
            risk_kof_stab = risk_kof_nostab

        risk_kof_stab_gate_tau = risk_kof_stab.copy()
        risk_kof_stab_gate_tau[conf_te < float(args.TAU_TARGET)] = -1e9

        res_kof_nostab = eval_monitoring_metrics(
            conf_te, err_te, risk_kof_nostab, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
        )
        res_kof_stab = eval_monitoring_metrics(
            conf_te, err_te, risk_kof_stab, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
        )
        res_kof_stab_gate = eval_monitoring_metrics(
            conf_te, err_te, risk_kof_stab_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
        )
        res_tcp = eval_monitoring_metrics(
            conf_te, err_te, risk_tcp, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
        )
        res_tcp_gate = eval_monitoring_metrics(
            conf_te, err_te, risk_tcp_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
        )
        res_rand_gate = eval_monitoring_metrics(
            conf_te, err_te, risk_rand_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False
        )
        res_tcpm = eval_monitoring_metrics(
            conf_te, err_te, risk_tcpm, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
        )
        res_tcpm_gate = eval_monitoring_metrics(
            conf_te, err_te, risk_tcpm_gate_tau, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
        )

        # NEW: tuned bounded evaluation for tcp_m_gate_tau only
        res_tcpm_gate_bounded_tuned = eval_monitoring_metrics_budgetwise_bounded(
            conf=conf_te,
            err=err_te,
            risk_score=risk_tcpm_gate_tau,
            group=g_te,
            budget_to_delta=tcpm_bounded_delta_by_budget,
            high_conf_region=args.HIGH_CONF_REGION,
            tau_target=args.TAU_TARGET,
            policy_name="bounded_tuned",
            return_alerted=True,
        )

        res_red = eval_monitoring_metrics(
            conf_te, err_te, risk_red, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False
        )
        res_ent = eval_monitoring_metrics(
            conf_te, err_te, risk_ent, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False
        )
        res_energy = eval_monitoring_metrics(
            conf_te, err_te, risk_energy, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
            policy="global", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=False
        )

        mitig = []
        if args.ENABLE_FAIRNESS_MITIGATION:
            base_masks = {
                "tcp_gate_tau": res_tcp_gate.get("alerted_masks", {}),
                "tcp_m_gate_tau": res_tcpm_gate.get("alerted_masks", {}),
                "kof_stab": res_kof_stab.get("alerted_masks", {}),
                "kof_stab_gate_tau": res_kof_stab_gate.get("alerted_masks", {}),
            }
            risk_map = {
                "tcp_gate_tau": risk_tcp_gate_tau,
                "tcp_m_gate_tau": risk_tcpm_gate_tau,
                "kof_stab": risk_kof_stab,
                "kof_stab_gate_tau": risk_kof_stab_gate_tau,
            }

            for mname in args.MITIGATE_MODELS:
                if mname not in risk_map:
                    continue
                rv = risk_map[mname]
                bm = base_masks.get(mname, {})

                if "parity" in args.FAIRNESS_POLICIES:
                    rr = eval_monitoring_metrics(
                        conf_te, err_te, rv, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
                        policy="parity", delta=0.0, tau_target=args.TAU_TARGET, return_alerted=True
                    )
                    mitig.append((mname, rr))
                    for b in args.ALERT_BUDGETS:
                        b = float(b)
                        if (b in rr.get("alerted_masks", {})) and (b in bm):
                            selchg_rows.append({
                                "exp": cfg.name,
                                "seed": seed,
                                "rho": float(rho),
                                "model": mname,
                                "policy": "parity",
                                "delta": -1.0,
                                "budget": b,
                                "n_selection_changed": int(np.sum(rr["alerted_masks"][b] != bm[b])),
                            })

                if "bounded" in args.FAIRNESS_POLICIES:
                    for dd in args.FAIR_TOPK_DELTAS:
                        rr = eval_monitoring_metrics(
                            conf_te, err_te, rv, g_te, args.ALERT_BUDGETS, args.HIGH_CONF_REGION,
                            policy="bounded", delta=float(dd), tau_target=args.TAU_TARGET, return_alerted=True
                        )
                        mitig.append((mname, rr))
                        for b in args.ALERT_BUDGETS:
                            b = float(b)
                            if (b in rr.get("alerted_masks", {})) and (b in bm):
                                selchg_rows.append({
                                    "exp": cfg.name,
                                    "seed": seed,
                                    "rho": float(rho),
                                    "model": mname,
                                    "policy": "bounded",
                                    "delta": float(dd),
                                    "budget": b,
                                    "n_selection_changed": int(np.sum(rr["alerted_masks"][b] != bm[b])),
                                })

        overall_rows.append({
            "exp": cfg.name,
            "seed": seed,
            "rho": float(rho),
            "high_conf_n": int(np.sum(conf_te >= float(args.HIGH_CONF_REGION))),
            "hc_auc_kof_nostab": res_kof_nostab["hc_auc"],
            "hc_auc_tau_kof_nostab": res_kof_nostab["hc_auc_tau"],
            "hc_auc_kof_stab": res_kof_stab["hc_auc"],
            "hc_auc_tau_kof_stab": res_kof_stab["hc_auc_tau"],
            "hc_auc_kof_stab_gate_tau": res_kof_stab_gate["hc_auc"],
            "hc_auc_tau_kof_stab_gate_tau": res_kof_stab_gate["hc_auc_tau"],
            "hc_auc_tcp": res_tcp["hc_auc"],
            "hc_auc_tau_tcp": res_tcp["hc_auc_tau"],
            "hc_auc_tcp_gate_tau": res_tcp_gate["hc_auc"],
            "hc_auc_tau_tcp_gate_tau": res_tcp_gate["hc_auc_tau"],
            "hc_auc_rand_gate_tau": res_rand_gate["hc_auc"],
            "hc_auc_tau_rand_gate_tau": res_rand_gate["hc_auc_tau"],
            "hc_auc_tcp_m": res_tcpm["hc_auc"],
            "hc_auc_tau_tcp_m": res_tcpm["hc_auc_tau"],
            "hc_auc_tcp_m_gate_tau": res_tcpm_gate["hc_auc"],
            "hc_auc_tau_tcp_m_gate_tau": res_tcpm_gate["hc_auc_tau"],
            "hc_auc_red": res_red["hc_auc"],
            "hc_auc_tau_red": res_red["hc_auc_tau"],
            "hc_auc_entropy": res_ent["hc_auc"],
            "hc_auc_tau_entropy": res_ent["hc_auc_tau"],
            "hc_auc_energy": res_energy["hc_auc"],
            "hc_auc_tau_energy": res_energy["hc_auc_tau"],
        })

        def append_model(model_name: str, res):
            for a in res["alerts"]:
                alert_rows.append({
                    "exp": cfg.name, "seed": seed, "rho": float(rho),
                    "model": model_name,
                    **a
                })
            for f in res["fairness"]:
                fair_rows.append({
                    "exp": cfg.name, "seed": seed, "rho": float(rho),
                    "model": model_name,
                    **f
                })
            for g in res["groups"]:
                group_rows.append({
                    "exp": cfg.name, "seed": seed, "rho": float(rho),
                    "model": model_name,
                    **g
                })

        append_model("kofnostab", res_kof_nostab)
        append_model("kofstab", res_kof_stab)
        append_model("kofstabgatetau", res_kof_stab_gate)
        append_model("tcp", res_tcp)
        append_model("tcpgatetau", res_tcp_gate)
        append_model("randgatetau", res_rand_gate)
        append_model("tcpm", res_tcpm)
        append_model("tcpmgatetau", res_tcpm_gate)
        append_model("tcpmgatetau_bounded_tuned", res_tcpm_gate_bounded_tuned)
        append_model("red", res_red)
        append_model("entropy", res_ent)
        append_model("energy", res_energy)

        for mname, rr in mitig:
            suffix = rr["alerts"][0]["policy"] if len(rr["alerts"]) else "mitig"
            dd = rr["alerts"][0]["delta"] if len(rr["alerts"]) else -1.0
            tag = f"{mname}__{suffix}__delta_{dd}"
            append_model(tag, rr)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    meta = {
        "seed": seed,
        "exp": cfg.name,
        "dataset": "ACSIncome",
        "years": args.YEARS,
        "train_states": args.TRAIN_STATES,
        "shift_states": args.SHIFT_STATES,
        "sensitive_col_eval_only": SENSITIVE_COL,
        "swap_col": swap_col,
        "perm_col": perm_col,
        "swap_map_size": len(swap_map),
        "perm_map_size": len(perm_map),
        "tau_target": args.TAU_TARGET,
        "tau_list": args.TAU_LIST,
        "high_conf_region": args.HIGH_CONF_REGION,
        "alert_budgets": args.ALERT_BUDGETS,
        "fairness_mitigation_enabled": int(args.ENABLE_FAIRNESS_MITIGATION),
        "fairness_policies": args.FAIRNESS_POLICIES,
        "fair_topk_deltas": args.FAIR_TOPK_DELTAS,
        "mitigate_models": args.MITIGATE_MODELS,
        "tcp_m_gate_tau_bounded_delta_by_budget": json.dumps(
            {str(float(k)): float(v) for k, v in tcpm_bounded_delta_by_budget.items()}
        ),
    }

    return (
        pd.DataFrame(overall_rows),
        pd.DataFrame(alert_rows),
        pd.DataFrame(fair_rows),
        pd.DataFrame(group_rows),
        pd.DataFrame(selchg_rows),
        meta,
        tcpm_bounded_delta_diag,
        pd.DataFrame(dataset_stats_rows),
    )

# -----------------------------
# Main
# -----------------------------
def main():
    ensure_outdirs(args.OUTDIR)
    TABLES = os.path.join(args.OUTDIR, "tables")
    FIGS = os.path.join(args.OUTDIR, "figs")

    versions = dump_versions()
    meta_rows = []
    all_overall, all_alerts, all_fair, all_groups, all_selchg = [], [], [], [], []

    # NEW: tuning-diagnostic collection
    all_tcpm_bounded_diag = []

    # NEW: reviewer-requested dataset/group statistics collection
    all_dataset_stats = []

    for cfg in experiments():
        for seed in args.SEEDS:
            out = run_one_seed_experiment(seed, cfg)

            # NEW: unpack eighth return value (dataset_stats)
            ov, al, fa, gr, sc, meta, tcpm_diag, dstats = out

            all_overall.append(ov)
            all_alerts.append(al)
            all_fair.append(fa)
            all_groups.append(gr)
            all_selchg.append(sc)
            meta_rows.append(meta)

            # NEW: save tuning diagnostics
            if tcpm_diag is not None and len(tcpm_diag):
                d = tcpm_diag.copy()
                d["exp"] = cfg.name
                d["seed"] = seed
                all_tcpm_bounded_diag.append(d)

            # NEW: save dataset/group statistics
            if dstats is not None and len(dstats):
                all_dataset_stats.append(dstats)

    raw_overall = pd.concat(all_overall, axis=0, ignore_index=True) if len(all_overall) else pd.DataFrame()
    raw_alerts = pd.concat(all_alerts, axis=0, ignore_index=True) if len(all_alerts) else pd.DataFrame()
    raw_fair = pd.concat(all_fair, axis=0, ignore_index=True) if len(all_fair) else pd.DataFrame()
    raw_groups = pd.concat(all_groups, axis=0, ignore_index=True) if len(all_groups) else pd.DataFrame()
    raw_selchg = pd.concat(all_selchg, axis=0, ignore_index=True) if len(all_selchg) else pd.DataFrame()
    meta_df = pd.DataFrame(meta_rows)

    # NEW
    raw_tcpm_bounded_diag = (
        pd.concat(all_tcpm_bounded_diag, axis=0, ignore_index=True)
        if len(all_tcpm_bounded_diag) else pd.DataFrame()
    )

    # Reviewer-requested dataset/group statistics table (consistent naming with diabetes script).
    raw_dataset_stats = (
        pd.concat(all_dataset_stats, axis=0, ignore_index=True)
        if len(all_dataset_stats) else pd.DataFrame()
    )
    raw_dataset_stats.to_csv(os.path.join(TABLES, "raw_dataset_shift_statistics.csv"), index=False)
    if len(raw_dataset_stats):
        dstat_metric_cols = [c for c in raw_dataset_stats.columns if c not in ("split", "rho", "exp", "seed")]
        dataset_stats_summary = mean_std_ci_over_seeds(
            raw_dataset_stats,
            group_cols=["split", "rho"],
            metric_cols=dstat_metric_cols,
            seed_col="seed",
            n_boot=args.BOOTSTRAP_N,
            alpha=args.BOOTSTRAP_ALPHA,
        )
    else:
        dataset_stats_summary = pd.DataFrame()
    dataset_stats_summary.to_csv(os.path.join(TABLES, "summary_dataset_shift_statistics_mean_std_ci.csv"), index=False)

    raw_overall.to_csv(os.path.join(TABLES, "raw_monitor_overall.csv"), index=False)
    raw_alerts.to_csv(os.path.join(TABLES, "raw_monitor_alerts.csv"), index=False)
    raw_fair.to_csv(os.path.join(TABLES, "raw_monitor_fairness_alerts.csv"), index=False)
    raw_groups.to_csv(os.path.join(TABLES, "raw_monitor_group_alerts.csv"), index=False)
    raw_selchg.to_csv(os.path.join(TABLES, "raw_mitigation_selection_changes.csv"), index=False)
    meta_df.to_csv(os.path.join(TABLES, "run_metadata.csv"), index=False)

    # NEW
    raw_tcpm_bounded_diag.to_csv(os.path.join(TABLES, "raw_tcpm_bounded_delta_tuning.csv"), index=False)

    overall_metric_cols = [c for c in raw_overall.columns if c.startswith("hc_auc")]
    overall_summary = mean_std_ci_over_seeds(
        raw_overall,
        group_cols=["exp", "rho"],
        metric_cols=overall_metric_cols,
        seed_col="seed",
        n_boot=args.BOOTSTRAP_N,
        alpha=args.BOOTSTRAP_ALPHA,
    )
    overall_summary.to_csv(os.path.join(TABLES, "summary_monitor_overall_mean_std_ci.csv"), index=False)

    alert_metric_cols = [
        "precision", "recall", "lift", "worst_precision", "worst_recall", "worst_fpr",
        "z_rate_high", "n_alert", "n_high",
        "n_groups_total", "n_groups_precision_defined", "n_groups_recall_defined", "n_groups_fpr_defined",
        "n_alert_tau", "n_alert_mid", "frac_alert_tau", "frac_alert_mid",
    ]
    alerts_summary = mean_std_ci_over_seeds(
        raw_alerts,
        group_cols=["exp", "rho", "model", "budget", "policy", "delta"],
        metric_cols=[c for c in alert_metric_cols if c in raw_alerts.columns],
        seed_col="seed",
        n_boot=args.BOOTSTRAP_N,
        alpha=args.BOOTSTRAP_ALPHA,
    )
    alerts_summary.to_csv(os.path.join(TABLES, "summary_monitor_alerts_mean_std_ci.csv"), index=False)

    fair_metric_cols = ["alert_rate_gap", "alert_rate_ratio", "gap_precision", "gap_recall", "gap_fpr", "equal_opportunity_diff", "equalized_odds_diff"]
    fair_summary = mean_std_ci_over_seeds(
        raw_fair,
        group_cols=["exp", "rho", "model", "budget", "policy", "delta"],
        metric_cols=[c for c in fair_metric_cols if c in raw_fair.columns],
        seed_col="seed",
        n_boot=args.BOOTSTRAP_N,
        alpha=args.BOOTSTRAP_ALPHA,
    )
    fair_summary.to_csv(os.path.join(TABLES, "summary_monitor_fairness_mean_std_ci.csv"), index=False)

    group_metric_cols = [
        "n_high_g", "n_alert_g", "n_z_g", "n_alert_and_z_g",
        "n_neg_g", "n_alert_and_negz_g",
        "alert_rate_g", "precision_g", "recall_g", "fpr_g"
    ]
    group_summary = mean_std_ci_over_seeds(
        raw_groups,
        group_cols=["exp", "rho", "model", "budget", "policy", "delta", "group"],
        metric_cols=[c for c in group_metric_cols if c in raw_groups.columns],
        seed_col="seed",
        n_boot=args.BOOTSTRAP_N,
        alpha=args.BOOTSTRAP_ALPHA,
    )
    group_summary.to_csv(os.path.join(TABLES, "summary_monitor_group_mean_std_ci.csv"), index=False)

    if len(raw_selchg):
        selchg_summary = mean_std_ci_over_seeds(
            raw_selchg,
            group_cols=["exp", "rho", "model", "budget", "policy", "delta"],
            metric_cols=["n_selection_changed"],
            seed_col="seed",
            n_boot=args.BOOTSTRAP_N,
            alpha=args.BOOTSTRAP_ALPHA,
        )
    else:
        selchg_summary = pd.DataFrame()
    selchg_summary.to_csv(os.path.join(TABLES, "summary_mitigation_selection_changes_mean_std_ci.csv"), index=False)

    # NEW: summarize bounded tuning diagnostics across seeds
    if len(raw_tcpm_bounded_diag):
        tcpm_bounded_diag_summary = mean_std_ci_over_seeds(
            raw_tcpm_bounded_diag,
            group_cols=["exp", "budget"],
            metric_cols=[
                c for c in [
                    "chosen_delta",
                    "base_precision", "base_recall", "base_lift",
                    "base_gap_precision", "base_gap_recall",
                    "chosen_precision", "chosen_recall", "chosen_lift",
                    "chosen_gap_precision", "chosen_gap_recall",
                ] if c in raw_tcpm_bounded_diag.columns
            ],
            seed_col="seed",
            n_boot=args.BOOTSTRAP_N,
            alpha=args.BOOTSTRAP_ALPHA,
        )
    else:
        tcpm_bounded_diag_summary = pd.DataFrame()

    tcpm_bounded_diag_summary.to_csv(
        os.path.join(TABLES, "summary_tcpm_bounded_delta_tuning.csv"),
        index=False
    )

    # Tradeoff table: join alerts + fairness (matches diabetes script's raw/summary_monitor_tradeoff naming)
    tradeoff_raw = raw_alerts.merge(
        raw_fair,
        on=["exp", "seed", "rho", "model", "budget", "policy", "delta"],
        how="inner"
    )
    tradeoff_raw.to_csv(os.path.join(TABLES, "raw_monitor_tradeoff.csv"), index=False)
    tradeoff_metric_cols = [
        "precision", "recall", "lift", "alert_rate_gap",
        "worst_precision", "worst_recall", "worst_fpr",
        "gap_precision", "gap_recall", "gap_fpr",
        "equal_opportunity_diff", "equalized_odds_diff", "z_rate_high",
    ]
    tradeoff_summary = mean_std_ci_over_seeds(
        tradeoff_raw,
        group_cols=["exp", "rho", "model", "budget", "policy", "delta"],
        metric_cols=[c for c in tradeoff_metric_cols if c in tradeoff_raw.columns],
        seed_col="seed",
        n_boot=args.BOOTSTRAP_N,
        alpha=args.BOOTSTRAP_ALPHA,
    )
    tradeoff_summary.to_csv(os.path.join(TABLES, "summary_monitor_tradeoff_mean_std_ci.csv"), index=False)

    # Filtered fairness tables by minimum mean high-conf group size
    for nmin in args.GROUP_MIN_N_LIST:
        gs = group_summary.copy()
        if len(gs):
            keep = gs["n_high_g_mean"] >= float(nmin)
            gs2 = gs[keep].copy()
        else:
            gs2 = gs
        gs2.to_csv(os.path.join(TABLES, f"summary_monitor_group_mean_std_ci_minN{int(nmin)}.csv"), index=False)

    # -----------------------------
    # Figures
    # -----------------------------
    BSEL = float(args.ALERT_BUDGETS[0])

    # Overall AUC curves
    auc_map = [
        ("hc_auc_kof_nostab", "kof_nostab"),
        ("hc_auc_kof_stab", "kof_stab"),
        ("hc_auc_kof_stab_gate_tau", "kof_stab_gate_tau"),
        ("hc_auc_tcp", "tcp"),
        ("hc_auc_tcp_gate_tau", "tcp_gate_tau"),
        ("hc_auc_rand_gate_tau", "rand_gate_tau"),
        ("hc_auc_tcp_m", "tcp_m"),
        ("hc_auc_tcp_m_gate_tau", "tcp_m_gate_tau"),
        ("hc_auc_red", "red_lr"),
        ("hc_auc_entropy", "entropy"),
        ("hc_auc_energy", "energy"),
    ]
    rows_auc = []
    for _, r in overall_summary.iterrows():
        for col, label in auc_map:
            if f"{col}_mean" in overall_summary.columns:
                rows_auc.append({
                    "exp": r["exp"], "rho": r["rho"], "model": label,
                    "mean": r.get(f"{col}_mean", np.nan),
                    "lo": r.get(f"{col}_ci_low", np.nan),
                    "hi": r.get(f"{col}_ci_high", np.nan),
                })
    auc_df = pd.DataFrame(rows_auc)
    if len(auc_df):
        sub = auc_df[auc_df["exp"] == "full"].copy()
        plot_line_ci(
            sub, "rho", "mean", "lo", "hi", "model",
            "High-confidence monitoring AUC vs rho (FULL)",
            "HC AUC",
            os.path.join(FIGS, "hc_auc_vs_rho_full.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red_lr", "entropy", "energy", "rand_gate_tau"],
        )

    # Alert precision / recall / lift for global policy
    subA = alerts_summary[
        (alerts_summary["exp"] == "full") &
        (np.isclose(alerts_summary["budget"], BSEL)) &
        (alerts_summary["policy"] == "global") &
        (alerts_summary["delta"] == -1.0)
    ].copy()
    if len(subA):
        p_df = subA[["rho", "model", "precision_mean", "precision_ci_low", "precision_ci_high"]].rename(
            columns={"precision_mean": "mean", "precision_ci_low": "lo", "precision_ci_high": "hi"}
        )
        plot_line_ci(
            p_df, "rho", "mean", "lo", "hi", "model",
            f"Alert precision vs rho @ budget={BSEL:.2f} (FULL, global)", "precision",
            os.path.join(FIGS, "alert_precision_vs_rho.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "tcpmgatetau_bounded_tuned", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red", "entropy", "energy"],
        )

        r_df = subA[["rho", "model", "recall_mean", "recall_ci_low", "recall_ci_high"]].rename(
            columns={"recall_mean": "mean", "recall_ci_low": "lo", "recall_ci_high": "hi"}
        )
        plot_line_ci(
            r_df, "rho", "mean", "lo", "hi", "model",
            f"Alert recall vs rho @ budget={BSEL:.2f} (FULL, global)", "recall",
            os.path.join(FIGS, "alert_recall_vs_rho.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "tcpmgatetau_bounded_tuned", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red", "entropy", "energy"],
        )

        l_df = subA[["rho", "model", "lift_mean", "lift_ci_low", "lift_ci_high"]].rename(
            columns={"lift_mean": "mean", "lift_ci_low": "lo", "lift_ci_high": "hi"}
        )
        plot_line_ci(
            l_df, "rho", "mean", "lo", "hi", "model",
            f"Alert lift vs rho @ budget={BSEL:.2f} (FULL, global)", "lift",
            os.path.join(FIGS, "alert_lift_vs_rho.png"),
            order=["tcp", "tcp_m", "tcp_m_gate_tau", "tcpmgatetau_bounded_tuned", "kof_stab", "kof_stab_gate_tau", "kof_nostab", "red"],
        )

    # Fairness gap curves
    subF = fair_summary[
        (fair_summary["exp"] == "full") &
        (np.isclose(fair_summary["budget"], BSEL))
    ].copy()
    if len(subF):
        for pol in ["global", "parity", "bounded_tuned"]:
            dpol = subF[subF["policy"] == pol].copy()
            dpol = dpol[dpol["model"].isin(["tcpmgatetau", "tcpmgatetau_bounded_tuned", "kofstabgatetau", "kofstab"])]
            if len(dpol):
                g_df = dpol[["rho", "model", "alert_rate_gap_mean", "alert_rate_gap_ci_low", "alert_rate_gap_ci_high"]].rename(
                    columns={"alert_rate_gap_mean": "mean", "alert_rate_gap_ci_low": "lo", "alert_rate_gap_ci_high": "hi"}
                )
                plot_line_ci(
                    g_df, "rho", "mean", "lo", "hi", "model",
                    f"Alert-rate gap vs rho @ budget={BSEL:.2f} ({pol})",
                    "alert-rate gap",
                    os.path.join(FIGS, f"fairness_gap_vs_rho_{pol}.png"),
                    order=["tcpmgatetau", "tcpmgatetau_bounded_tuned", "kofstabgatetau", "kofstab"],
                )

    # Pareto scatter
    RHO_SEL = 1.0
    pareto = tradeoff_summary[
        (tradeoff_summary["exp"] == "full") &
        (np.isclose(tradeoff_summary["rho"], RHO_SEL)) &
        (np.isclose(tradeoff_summary["budget"], BSEL)) &
        (tradeoff_summary["model"].isin(["tcp", "tcpm", "tcpmgatetau", "tcpmgatetau_bounded_tuned", "kofstab", "kofstabgatetau"]))
    ].copy()
    if len(pareto):
        def _lab2(row):
            if row["policy"] == "bounded":
                return f"{row['model']}/bounded(d={row['delta']:.2f})"
            if row["policy"] == "bounded_tuned":
                return f"{row['model']}/bounded_tuned"
            return f"{row['model']}/{row['policy']}"

        pareto["label"] = pareto.apply(_lab2, axis=1)

        plt.figure(figsize=(8, 6))
        for _, r in pareto.iterrows():
            x = float(r["alert_rate_gap_mean"])
            y = float(r["precision_mean"])
            xlo = float(r["alert_rate_gap_ci_low"])
            xhi = float(r["alert_rate_gap_ci_high"])
            ylo = float(r["precision_ci_low"])
            yhi = float(r["precision_ci_high"])
            plt.scatter([x], [y])
            plt.errorbar([x], [y], xerr=[[x - xlo], [xhi - x]], yerr=[[y - ylo], [yhi - y]], fmt="none", capsize=3)
            plt.text(x, y, str(r["label"]), fontsize=8, ha="left", va="bottom")
        plt.xlabel("alert-rate gap")
        plt.ylabel("overall alert precision")
        plt.title(f"Pareto: precision vs fairness gap @ rho={int(RHO_SEL)}, budget={BSEL:.2f}")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(FIGS, f"pareto_precision_vs_gap_rho{int(RHO_SEL)}_budget{BSEL:.2f}.png"), dpi=160)
        plt.close()

    # -----------------------------
    # Report
    # -----------------------------
    report_md = os.path.join(args.OUTDIR, "report.md")
    with open(report_md, "w", encoding="utf-8") as f:
        f.write("# Protocol-valid fair auditing of high-confidence failures under distribution shift (ACSIncome)\n\n")
        f.write("## Framing\n")
        f.write(f"- Target label: z=1[err & conf>={args.TAU_TARGET:.2f}]\n")
        f.write(f"- Monitoring slice: conf>={args.HIGH_CONF_REGION:.2f}\n")
        f.write("- Allocation policies: global / parity / bounded(delta)\n")
        f.write("- Budgeting: K=floor(budget*n), allows K=0\n")
        f.write("- Lift: precision / P(z=1 | monitored slice)\n\n")

        f.write("## Versions\n```json\n")
        f.write(json.dumps(versions, indent=2))
        f.write("\n```\n\n")

        f.write("## Args\n```json\n")
        f.write(json.dumps(asdict(args), indent=2))
        f.write("\n```\n\n")

        f.write("## Summary: Overall (mean/std/CI)\n")
        f.write(overall_summary.to_markdown(index=False))
        f.write("\n\n")

        f.write("## Summary: Alerts (mean/std/CI)\n")
        f.write(alerts_summary.to_markdown(index=False))
        f.write("\n\n")

        f.write("## Summary: Fairness (mean/std/CI)\n")
        f.write(fair_summary.to_markdown(index=False))
        f.write("\n\n")

        f.write("## Summary: Tradeoff (mean/std/CI)\n")
        f.write(tradeoff_summary.to_markdown(index=False))
        f.write("\n\n")

        # NEW
        if len(tcpm_bounded_diag_summary):
            f.write("## Summary: TCP-M bounded-delta tuning\n")
            f.write(tcpm_bounded_diag_summary.to_markdown(index=False))
            f.write("\n\n")

        if len(raw_selchg):
            f.write("## Selection-only confirmation\n")
            f.write("Mitigation changes alert allocation only (scores unchanged). Rows show #changed alerts vs global.\n\n")
            f.write(raw_selchg.head(200).to_markdown(index=False))
            f.write("\n\n")

        f.write("## Figures\n")
        for fn in sorted(os.listdir(FIGS)):
            f.write(f"- figs/{fn}\n")

    zip_path = args.OUTDIR.rstrip("/").rstrip("\\") + ".zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(args.OUTDIR):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, args.OUTDIR)
                z.write(full, arcname=rel)

    print("\nDONE.")
    print("OUTDIR:", args.OUTDIR)
    print("Zip:", zip_path)


if __name__ == "__main__":
    main()